# ScoutRAG: Constrained Hybrid Retrieval for Football Player Scouting

**CS 455 -- Large Language Models -- Spring 2025/2026**

**Group:** Mustafa Ege Ozer | Umut Can Cubukcu | Yigit Onur Yilmaz

**Track:** CS 455 Standard Project

---

> **Framing note.** This notebook implements a *constrained hybrid retrieval* system
> over structured and semantic player attributes. It is **not** a Text-to-SQL system
> -- no executable SQL is generated. Explicit numerical/categorical constraints are
> applied through Pandas filtering; semantic and tactical requirements are resolved
> with sentence-transformer embeddings and cosine similarity.
>
> **Data disclaimer.** The dataset is derived from Football Manager 2024 (FM24),
> a commercial football-simulation game by Sports Interactive / SEGA. Player ratings
> are *designer-set game attributes*, not validated real-world scouting measurements.
> This dataset serves as a **synthetic / game-derived prototype-validation corpus**.
> It is appropriate for demonstrating the retrieval pipeline in a course setting, but
> results cannot be generalised to real scouting practice, and the dataset should not
> be redistributed for academic publication without clarifying its licensed-software origin.

---
## Section 1 -- Project Framing and Scope

### What we are building
ScoutRAG is a **hybrid structured-symbolic and semantic retrieval** assistant for
football player scouting. Given a natural-language scouting query (e.g.
*"left-footed centre-back under 23 with strong passing and tackling, valued under
EUR 10M"*), the system:

1. **Parses** explicit constraints (age, position, foot, value, wage, attribute thresholds).
2. **Filters** the player database using those constraints via Pandas -- verifiable, exact matches.
3. **Embeds** the residual semantic / tactical part of the query and retrieves the most
   similar player profiles using cosine similarity over sentence-transformer embeddings.
4. **Combines** both signals into a ranked shortlist.
5. **Generates** a grounded scouting report whose every cited value comes from a retrieved row.
6. **Verifies** the report: every cited player name and numeric attribute is checked against
   the source rows, and discrepancies are flagged as potential hallucinations.

### What the system is NOT
- It does **not** generate or execute SQL queries.
- It does **not** use an LLM as a black-box oracle; generation is grounded strictly on retrieved rows.
- It does **not** claim real-world scouting validity -- FM24 ratings are a synthetic proxy.

### Academic contribution (CS 455 standard track)
The most technically novel part is the **post-generation grounding-verification step**:
an automated check that computes a hallucination / grounding score by comparing every
numeric claim in the generated report against source structured rows. This is the core
contribution developed and evaluated carefully throughout the notebook.

---
## Section 2 -- Imports and Setup

Install and import all dependencies. Designed for **Google Colab (free tier)** or local Python.
FAISS is attempted first; if it fails, sklearn cosine_similarity is the fallback.

In [ ]:
# -- Google Drive mount -------------------------------------------------------
# Run this cell first. It will open a Google authorisation prompt.
# After mounting, your Drive is accessible at /content/drive/MyDrive/
# Upload fmdata24llm.csv and player_embeddings.pkl to your Drive root
# (or change the DRIVE_ROOT path below to a subfolder).

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'   # change if your files are in a subfolder
print(f'Drive mounted. Root: {DRIVE_ROOT}')

# Quick check: confirm the dataset file is present
csv_path = os.path.join(DRIVE_ROOT, 'fmdata24llm.csv')
pkl_path = os.path.join(DRIVE_ROOT, 'player_embeddings.pkl')
print(f'Dataset found : {os.path.exists(csv_path)}  ({csv_path})')
print(f'Embeddings pkl: {os.path.exists(pkl_path)}  ({pkl_path})')


In [1]:
# -- Install dependencies (run once; comment out if already installed) ----------
import subprocess
import sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

pip_install("sentence-transformers")
pip_install("scikit-learn")
pip_install("pandas")
pip_install("numpy")
pip_install("matplotlib")
pip_install("seaborn")

# Try FAISS (faiss-gpu is conda-only; faiss-cpu works everywhere)
import importlib
if importlib.util.find_spec("faiss") is None:
    pip_install("faiss-cpu")
print("All dependencies ready.")

All dependencies ready.


In [ ]:
import os, re, json, pickle, warnings, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

FAISS_AVAILABLE = False
try:
    import faiss
    FAISS_AVAILABLE = True
    print(f"FAISS available (version {faiss.__version__})")
except ImportError:
    print("FAISS not available -- using sklearn fallback")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__}")


# -- Configuration -----------------------------------------------------------
DRIVE_ROOT = '/content/drive/MyDrive'  # must match the cell above

DATA_PATH  = os.path.join(DRIVE_ROOT, 'fmdata24llm.csv')

# -- Load --------------------------------------------------------------------
---
## Section 3 -- Load and Inspect Dataset

### Dataset: Football Manager 2024 (FM24) export
- **Source:** In-game export from FM24 (Sports Interactive / SEGA), attribute masking disabled.
- **Size:** 42,541 player profiles, 58 columns.
- **Nature:** Game-derived / synthetic data. Ratings are designer-assigned integers on 1-20 scale.
- **Limitation:** Values, wages, and transfer info reflect FM24's simulated economy, not real-world
  finances. Nationality and club are embedded inside the `Name` and `Club` columns as suffixes.

> **Colab note:** Upload `fmdata24llm.csv` to your session or mount Google Drive, then
> adjust `DATA_PATH` below.

In [3]:
# -- Configuration -- adjust DATA_PATH to your environment ---------------------
DATA_PATH = "fmdata24llm.csv"          # local / Colab upload
# DATA_PATH = "/content/drive/MyDrive/fmdata24llm.csv"  # Google Drive

# -- Load ----------------------------------------------------------------------
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Dataset loaded: 42,541 rows x 58 columns


In [4]:
# -- Column inventory ----------------------------------------------------------
print("All columns:")
for i, col in enumerate(df_raw.columns):
    dtype = df_raw[col].dtype
    n_missing = df_raw[col].isna().sum()
    pct = 100 * n_missing / len(df_raw)
    sample = df_raw[col].dropna().iloc[0] if n_missing < len(df_raw) else "ALL NULL"
    print(f"  [{i:2d}] {col:<18} dtype={str(dtype):<8}  "
          f"missing={n_missing:>5} ({pct:4.1f}%)  sample={repr(str(sample)[:40])}")

All columns:
  [ 0] Name               dtype=object    missing=    0 ( 0.0%)  sample='Lionel Messi - Argentinian'
  [ 1] Inf                dtype=object    missing=23647 (55.6%)  sample='Inj'
  [ 2] Age                dtype=int64     missing=    0 ( 0.0%)  sample='36'
  [ 3] Weight             dtype=object    missing=    0 ( 0.0%)  sample='67 kg'
  [ 4] Club               dtype=object    missing=    0 ( 0.0%)  sample='Inter Miami - Major League Soccer'
  [ 5] Style              dtype=object    missing=    0 ( 0.0%)  sample='Creative'
  [ 6] Preferred Foot     dtype=object    missing=    0 ( 0.0%)  sample='Left'
  [ 7] Position           dtype=object    missing= 1895 ( 4.5%)  sample='AM (RC), ST (C)'
  [ 8] Transfer Value     dtype=object    missing=    0 ( 0.0%)  sample='Not for Sale'
  [ 9] Wage               dtype=object    missing=  887 ( 2.1%)  sample='€1,591,000 p/m'
  [10] Wor                dtype=int64     missing=    0 ( 0.0%)  sample='9'
  [11] Vis                dtype=int64  

In [5]:
# -- Sample rows ---------------------------------------------------------------
df_raw.head(3)

,Name,Inf,Age,Weight,Club,Style,Preferred Foot,Position,Transfer Value,Wage,Wor,Vis,Thr,Tec,Tea,Tck,Str,Sta,TRO,Ref,Pun,Pos,Pen,Pas,Pac,1v1,OtB,Nat,Mar,L Th,Lon,Ldr,Kic,Jum,Hea,Han,Fre,Fla,Fir,Fin,Ecc,Dri,Det,Dec,Cro,Cor,Cnt,Cmp,Com,Cmd,Bra,Bal,Ant,Agi,Agg,Aer,Acc,Rec
0,Lionel Messi - Argentinian,NaN,36,67 kg,Inter Miami - Major League Soccer,Creative,Left,"AM (RC), ST (C)",Not for Sale,"€1,591,000 p/m",9,20,2,20,14,7,9,13,3,1,4,5,17,19,15,1,14,14,4,4,16,14,3,6,10,3,18,20,19,17,1,20,20,18,15,15,13,16,2,1,10,18,16,15,7,4,16,- - -
1,Kevin De Bruyne - Belgian,Inj,32,68 kg,Man City - English Premier Division,Creative,Right,"M (RLC), AM (C)",Not for Sale,"€1,718,000 p/m",15,20,2,18,14,9,13,16,1,3,1,10,15,18,14,3,15,16,9,7,17,13,2,10,6,2,17,16,16,16,2,16,17,18,19,14,15,14,3,3,13,14,14,13,12,2,16,- - -
2,Kylian Mbappé - French,NaN,24,73 kg,Paris SG - Ligue 1 Uber Eats,Technical,Right,"AM (RL), ST (C)",€146M - €220M,"€4,488,000 p/m",12,15,3,17,10,4,11,14,2,1,1,3,18,15,20,3,17,15,4,3,15,13,3,8,8,2,11,18,18,17,3,18,18,15,12,13,14,17,3,3,12,14,17,16,6,3,20,- - -


In [6]:
# -- Column categorisation -----------------------------------------------------

# Identity columns
IDENTITY_COLS = ["Name", "Club", "Age", "Position", "Preferred Foot", "Style",
                 "Transfer Value", "Wage", "Weight", "Inf"]

# All 1-20 numeric attributes (outfield + GK)
ALL_ATTR_COLS = [
    "Acc", "Agi", "Bal", "Bra", "Cmp", "Cnt", "Cor", "Cro",
    "Dec", "Det", "Dri", "Fin", "Fir", "Fla", "Fre", "Hea",
    "Jum", "Ldr", "Lon", "Mar", "Nat", "OtB", "Pac", "Pas",
    "Pen", "Pos", "Sta", "Str", "Tck", "Tea", "Tec", "Vis", "Wor",
    "Agg", "Ant", "Aer",
    # GK-specific
    "1v1", "Com", "Cmd", "Ecc", "Han", "Kic", "L Th", "Pun", "Ref", "TRO", "Thr",
]

# GK-only attributes
GK_ONLY_COLS = ["1v1", "Com", "Cmd", "Ecc", "Han", "Kic", "L Th", "Pun", "Ref", "TRO", "Thr"]

# Outfield-relevant attributes (used for outfield player scoring)
OUTFIELD_ATTR_COLS = [c for c in ALL_ATTR_COLS if c not in GK_ONLY_COLS]

# Useless column (all values are "- -  -")
DROP_COLS = ["Rec"]

print(f"Total attribute columns : {len(ALL_ATTR_COLS)}")
print(f"Outfield attributes     : {len(OUTFIELD_ATTR_COLS)}")
print(f"GK-specific attributes  : {len(GK_ONLY_COLS)}")
print(f"Columns to drop         : {DROP_COLS}")

Total attribute columns : 47
Outfield attributes     : 36
GK-specific attributes  : 11
Columns to drop         : ['Rec']


In [7]:
# -- Dataset limitations summary -----------------------------------------------
print("=" * 60)
print("DATASET LIMITATIONS")
print("=" * 60)
limitations = [
    "1. NAME encodes nationality as suffix ('Player - Nationality').",
    "   No standalone 'Nationality' column exists.",
    "2. CLUB encodes league as suffix ('Club - League').",
    "   No standalone 'League' column exists.",
    "3. TRANSFER VALUE: string-encoded (e.g. 'EUR14K - EUR140K', 'Not for Sale').",
    "   Ranges require parsing; 'Not for Sale' is non-numeric.",
    "4. WAGE: string-encoded ('EUR1,500 p/m'); '-' for ~10.6% of players.",
    "5. INF: status code (Inj=Injured, Yth=Youth, Loa=Loan, Wnt=Wanted, etc.).",
    "   Missing for 55.6% of players -- only present when a status applies.",
    "6. REC column: all values are '- -  -' -- no information, will be dropped.",
    "7. POSITION: FM shorthand strings with 485 unique combinations.",
    "   Requires keyword-based mapping for natural-language queries.",
    "8. All numeric attributes are on a 1-20 integer scale (designer-set).",
    "   These are game ratings, not real-world measurements.",
    "9. Some players lack a position entry (1,895 missing).",
    "10. All data reflects FM24 game state -- salaries/values are fictional.",
]
for line in limitations:
    print(f"  {line}")

DATASET LIMITATIONS
  1. NAME encodes nationality as suffix ('Player - Nationality').
     No standalone 'Nationality' column exists.
  2. CLUB encodes league as suffix ('Club - League').
     No standalone 'League' column exists.
  3. TRANSFER VALUE: string-encoded (e.g. 'EUR14K - EUR140K', 'Not for Sale').
     Ranges require parsing; 'Not for Sale' is non-numeric.
  4. WAGE: string-encoded ('EUR1,500 p/m'); '-' for ~10.6% of players.
  5. INF: status code (Inj=Injured, Yth=Youth, Loa=Loan, Wnt=Wanted, etc.).
     Missing for 55.6% of players -- only present when a status applies.
  6. REC column: all values are '- -  -' -- no information, will be dropped.
  7. POSITION: FM shorthand strings with 485 unique combinations.
     Requires keyword-based mapping for natural-language queries.
  8. All numeric attributes are on a 1-20 integer scale (designer-set).
     These are game ratings, not real-world measurements.
  9. Some players lack a position entry (1,895 missing).
  10. All data

---
## Section 4 -- Data Cleaning and Normalization

Every cleaning step is explained. We never silently drop data.

In [8]:
# -- Working copy --------------------------------------------------------------
df = df_raw.drop(columns=DROP_COLS).copy()
print(f"Working dataframe: {df.shape[0]:,} rows x {df.shape[1]} columns")

Working dataframe: 42,541 rows x 57 columns


In [ ]:
# -- 4.1  Parse Player_Name and Nationality from 'Name' ------------------------
# Format: "Lionel Messi - Argentinian"
# Split on ' - ': last token = nationality, rest = player name.

df["Player_Name"] = df["Name"].apply(
    lambda x: " - ".join(str(x).split(" - ")[:-1]) if " - " in str(x) else str(x)
)
df["Nationality"] = df["Name"].apply(
    lambda x: str(x).split(" - ")[-1].strip() if " - " in str(x) else "Unknown"
)

# -- 4.2  Drop phantom rows ---------------------------------------------------
# 1,895 rows have Name="- -": no real player, no position, no club.
# They are empty engine slots from the FM24 export.
# Keeping them causes the retrieval pipeline to return nameless results,
# which the LLM then hallucinates names for.
#
# Safe filter: Nationality=="Unknown" AND Position is NaN
# (all real players have at least a nationality parsed from the Name field)

before = len(df)
phantom_mask = (df["Nationality"] == "Unknown") & (df["Position"].isna())
df = df[~phantom_mask].reset_index(drop=True)
dropped = before - len(df)
print(f"Dropped {dropped:,} phantom rows (no name, no position).")
print(f"Clean dataset: {len(df):,} players remaining.")

print("\nNationality value counts (top 10):")
print(df["Nationality"].value_counts().head(10))


In [10]:
# -- 4.2  Parse Club_Name and League from 'Club' -------------------------------
# Format: "Man City - English Premier Division"

df["Club_Name"] = df["Club"].apply(
    lambda x: " - ".join(str(x).split(" - ")[:-1]).strip() if " - " in str(x) else str(x).strip()
)
df["League"] = df["Club"].apply(
    lambda x: str(x).split(" - ")[-1].strip() if " - " in str(x) else "Unknown"
)

print("Top 10 leagues:")
print(df["League"].value_counts().head(10))

Top 10 leagues:
League
Unknown                     5851
Italian Serie A             1830
Sky Bet Championship        1456
English Premier Division    1359
Ligue 1 Uber Eats           1322
Italian Serie B             1279
Ligue 2 BKT                 1263
Bundesliga                  1155
Sky Bet League One          1136
3. Liga                     1032
Name: count, dtype: int64


In [11]:
# -- 4.3  Parse Transfer Value to numeric (EUR) --------------------------------
# Examples: 'EUR14K - EUR140K', 'EUR0', 'Not for Sale', 'EUR1.2M'
# Strategy: ranges -> take UPPER bound. 'Not for Sale' -> NaN (flagged separately).

def parse_eur_string(s):
    """Return float in EUR, or np.nan if non-numeric."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s in ("Not for Sale", "-", ""):
        return np.nan
    if s == "0":
        return 0.0
    # Strip currency symbol and spaces
    s = s.replace("EUR", "").replace("€", "").replace(",", "").strip()
    # If range, take the upper value
    if " - " in s:
        s = s.split(" - ")[-1].strip()
    m = re.match(r"^([0-9.]+)([KMB]?)$", s)
    if m:
        num = float(m.group(1))
        suffix = m.group(2)
        if suffix == "K":   num *= 1_000
        elif suffix == "M": num *= 1_000_000
        elif suffix == "B": num *= 1_000_000_000
        return num
    return np.nan

df["Value_EUR"] = df["Transfer Value"].apply(parse_eur_string)
df["Not_For_Sale"] = df["Transfer Value"].apply(lambda x: str(x).strip() == "Not for Sale")

print("Transfer Value parsing:")
print(f"  Parsed numeric values : {df['Value_EUR'].notna().sum():,}")
print(f"  'Not for Sale'        : {df['Not_For_Sale'].sum():,}")
print(f"  Unparseable / NaN     : {(df['Value_EUR'].isna() & ~df['Not_For_Sale']).sum():,}")

Transfer Value parsing:
  Parsed numeric values : 41,802
  'Not for Sale'        : 739
  Unparseable / NaN     : 0


In [12]:
# -- 4.4  Parse Wage to numeric (EUR per month) --------------------------------
# Examples: 'EUR1,500 p/m', 'EUR1,591,000 p/m', '-'

def parse_wage(w):
    """Return monthly wage in EUR as float, or NaN if missing."""
    if pd.isna(w) or str(w).strip() in ("-", ""):
        return np.nan
    w = str(w).replace("€", "").replace(",", "").replace(" p/m", "").strip()
    m = re.match(r"^([0-9.]+)([KMB]?)$", w)
    if m:
        num = float(m.group(1))
        suffix = m.group(2)
        if suffix == "K":   num *= 1_000
        elif suffix == "M": num *= 1_000_000
        return num
    return np.nan

df["Wage_EUR_pm"] = df["Wage"].apply(parse_wage)
print("Wage parsing:")
print(f"  Parsed numeric wages  : {df['Wage_EUR_pm'].notna().sum():,}")
print(f"  Missing / '-'         : {df['Wage_EUR_pm'].isna().sum():,}")

Wage parsing:
  Parsed numeric wages  : 37,124
  Missing / '-'         : 5,417


In [13]:
# -- 4.5  Parse Weight to numeric (kg) ----------------------------------------
df["Weight_kg"] = df["Weight"].apply(
    lambda x: int(re.sub(r"[^0-9]", "", str(x))) if pd.notna(x) and re.sub(r"[^0-9]", "", str(x)) else np.nan
)
print(f"Weight_kg: min={df['Weight_kg'].min()}, max={df['Weight_kg'].max()}, "
      f"missing={df['Weight_kg'].isna().sum()}")

Weight_kg: min=50, max=115, missing=0


In [14]:
# -- 4.6  Preferred Foot -- normalise to 3 categories -------------------------
# Original: Right, Right Only, Left, Left Only, Either -> collapsed to: Right, Left, Either

FOOT_MAP = {
    "Right":      "Right",
    "Right Only": "Right",
    "Left":       "Left",
    "Left Only":  "Left",
    "Either":     "Either",
}
df["Foot"] = df["Preferred Foot"].map(FOOT_MAP).fillna("Unknown")
print(df["Foot"].value_counts())

Foot
Right     32626
Left       9769
Either      146
Name: count, dtype: int64


In [15]:
# -- 4.7  Position -- keep original + extract primary position token -----------
# FM positions: "AM (RC), ST (C)", "D (C)", "GK"
# We keep the raw string AND extract the first position code for simple filtering.

def extract_primary_position(pos_str):
    """Return first position code from FM position string, e.g. 'AM' from 'AM (RC), ST (C)'."""
    if pd.isna(pos_str):
        return "Unknown"
    first = str(pos_str).split(",")[0].strip()
    base = re.split(r"\s*\(", first)[0].strip()
    return base

df["Primary_Pos"] = df["Position"].apply(extract_primary_position)
print("Primary position distribution (top 15):")
print(df["Primary_Pos"].value_counts().head(15))

Primary position distribution (top 15):
Primary_Pos
D            10421
AM            5279
DM            4662
GK            4457
M             4020
ST            4005
M/AM          3390
D/WB          2585
Unknown       1895
D/WB/M         839
D/WB/M/AM      440
WB             227
WB/M/AM        126
D/M             53
WB/M            51
Name: count, dtype: int64


In [16]:
# -- 4.8  Inf (status) -- encode as readable flags ----------------------------
INF_DECODE = {
    "Inj": "Injured",   "Wnt": "Transfer Listed", "Loa": "On Loan",
    "FrA": "Free Agent","Yth": "Youth",            "Ret": "Retired",
    "Trn": "Training",  "Sus": "Suspended",        "Lst": "Listed",
    "Opt": "Option",    "Una": "Unavailable",      "ESC": "Escape Clause",
    "Ama": "Amateur",   "Ctr": "Contract",         "Dnt": "Dont Renew",
    "Wp":  "Work Permit","nEU": "Non-EU",           "IPR": "In Progress",
    "Frt": "Feud",
}
df["Status"] = df["Inf"].map(INF_DECODE).fillna("Available")
print("Player status distribution (top 10):")
print(df["Status"].value_counts().head(10))

Player status distribution (top 10):
Status
Available          23647
Youth               7412
Transfer Listed     3517
Free Agent          3043
On Loan             2515
Injured             1047
Amateur              573
Listed               331
Contract             141
Non-EU                91
Name: count, dtype: int64


In [17]:
# -- 4.9  Numeric attribute check ----------------------------------------------
# Confirm all attribute columns are int64 and in [1, 20]
attr_ok = True
for col in ALL_ATTR_COLS:
    lo, hi = df[col].min(), df[col].max()
    if lo < 1 or hi > 20:
        print(f"  WARNING: {col} out of range [{lo}, {hi}]")
        attr_ok = False
if attr_ok:
    print("All attribute columns are integer-valued in [1, 20]")
n_missing_attr = df[ALL_ATTR_COLS].isna().sum().sum()
print(f"Total missing values in attribute columns: {n_missing_attr}")

All attribute columns are integer-valued in [1, 20]
Total missing values in attribute columns: 0


In [18]:
# -- 4.10  Handle missing Position --------------------------------------------
n_missing_pos = df["Position"].isna().sum()
print(f"Players with missing Position: {n_missing_pos:,} ({100*n_missing_pos/len(df):.1f}%)")
# Fill with 'Unknown' -- do NOT drop these players; their attributes are valid for semantic retrieval.
df["Position"] = df["Position"].fillna("Unknown")
df["Primary_Pos"] = df["Primary_Pos"].fillna("Unknown")

print(f"\nFinal clean dataframe: {df.shape[0]:,} rows x {df.shape[1]} columns")
new_cols = ["Player_Name", "Nationality", "Club_Name", "League",
            "Value_EUR", "Not_For_Sale", "Wage_EUR_pm", "Weight_kg", "Foot", "Primary_Pos", "Status"]
print("New derived columns:", new_cols)

Players with missing Position: 1,895 (4.5%)

Final clean dataframe: 42,541 rows x 68 columns
New derived columns: ['Player_Name', 'Nationality', 'Club_Name', 'League', 'Value_EUR', 'Not_For_Sale', 'Wage_EUR_pm', 'Weight_kg', 'Foot', 'Primary_Pos', 'Status']


In [19]:
# -- 4.11  Clean dataframe preview --------------------------------------------
display_cols = ["Player_Name", "Nationality", "Age", "Club_Name", "League",
                "Position", "Foot", "Style", "Value_EUR", "Wage_EUR_pm",
                "Acc", "Pac", "Pas", "Tec", "Sta", "Str", "Tck", "Dri", "Fin"]
df[display_cols].head(5)

,Player_Name,Nationality,Age,Club_Name,League,Position,Foot,Style,Value_EUR,Wage_EUR_pm,Acc,Pac,Pas,Tec,Sta,Str,Tck,Dri,Fin
0,Lionel Messi,Argentinian,36,Inter Miami,Major League Soccer,"AM (RC), ST (C)",Left,Creative,NaN,1591000.0,16,15,19,20,13,9,7,20,17
1,Kevin De Bruyne,Belgian,32,Man City,English Premier Division,"M (RLC), AM (C)",Right,Creative,NaN,1718000.0,16,14,18,18,16,13,9,16,16
2,Kylian Mbappé,French,24,Paris SG,Ligue 1 Uber Eats,"AM (RL), ST (C)",Right,Technical,220000000.0,4488000.0,20,20,15,17,14,11,4,18,17
3,Rodri,Spanish,27,Man City,English Premier Division,"D (C), DM, M (C)",Right,Intelligent,322000000.0,834000.0,14,13,16,14,17,16,17,13,13
4,Erling Haaland,Norwegian,22,Man City,English Premier Division,ST (C),Left,Physical,NaN,1718000.0,17,19,13,15,14,17,7,14,18


---
## Section 5 -- Player Profile Construction

Each player row is converted to a **compact natural-language profile** for
sentence-transformer embedding. The profile includes only information present in
the row -- no inference or hallucination. A `player_id` links every profile back
to the exact structured row.

In [20]:
# -- Attribute abbreviation -> readable name mapping --------------------------
ATTR_READABLE = {
    "Acc": "acceleration", "Agi": "agility",      "Bal": "balance",
    "Bra": "bravery",      "Cmp": "composure",    "Cnt": "concentration",
    "Cor": "corners",      "Cro": "crossing",     "Dec": "decisions",
    "Det": "determination","Dri": "dribbling",    "Fin": "finishing",
    "Fir": "first touch",  "Fla": "flair",        "Fre": "free kicks",
    "Hea": "heading",      "Jum": "jumping",      "Ldr": "leadership",
    "Lon": "long shots",   "Mar": "marking",      "Nat": "natural fitness",
    "OtB": "off the ball", "Pac": "pace",         "Pas": "passing",
    "Pen": "penalties",    "Pos": "positioning",  "Sta": "stamina",
    "Str": "strength",     "Tck": "tackling",     "Tea": "teamwork",
    "Tec": "technique",    "Vis": "vision",       "Wor": "work rate",
    "Agg": "aggression",   "Ant": "anticipation", "Aer": "aerial",
    # GK
    "1v1": "one-vs-one",   "Com": "communication","Cmd": "command of area",
    "Ecc": "eccentricity", "Han": "handling",     "Kic": "kicking",
    "L Th":"long throws",  "Pun": "punching",     "Ref": "reflexes",
    "TRO": "rush-out tendency","Thr": "throwing",
}


def build_profile(row):
    """Build a grounded text profile for one player row."""
    name   = row["Player_Name"]
    nat    = row["Nationality"]
    age    = row["Age"]
    pos    = row["Position"] if row["Position"] != "Unknown" else "unknown position"
    club   = row["Club_Name"]
    league = row["League"]
    foot   = row["Foot"]
    style  = row["Style"]
    status = row["Status"]

    val_str  = f"EUR{row['Value_EUR']:,.0f}" if pd.notna(row["Value_EUR"]) else "value unknown"
    if row["Not_For_Sale"]:
        val_str = "not for sale"
    wage_str = f"EUR{row['Wage_EUR_pm']:,.0f}/month" if pd.notna(row["Wage_EUR_pm"]) else "wage unknown"

    # Top 8 attributes by value (use GK attrs for GKs, outfield attrs for others)
    is_gk = "GK" in str(row["Position"])
    attr_pool = GK_ONLY_COLS if is_gk else OUTFIELD_ATTR_COLS
    attr_vals = {col: row[col] for col in attr_pool if col in row.index}
    top_attrs = sorted(attr_vals.items(), key=lambda x: -x[1])[:8]
    attr_str  = ", ".join(f"{ATTR_READABLE.get(k, k)} {v}" for k, v in top_attrs)

    profile = (
        f"{name} is a {nat} player aged {age}, playing as {pos} "
        f"for {club} ({league}). "
        f"Preferred foot: {foot}. Playing style: {style}. "
        f"Status: {status}. "
        f"Transfer value: {val_str}. Wage: {wage_str}. "
        f"Key attributes: {attr_str}."
    )
    return profile


# Build profiles for all players
print("Building player profiles...")
df["profile"]   = df.apply(build_profile, axis=1)
df["player_id"] = df.index  # stable integer ID

print(f"Profiles built: {len(df):,}")
print("\nExample profile (row 0):")
print(df["profile"].iloc[0])

Building player profiles...
Profiles built: 42,541

Example profile (row 0):
Lionel Messi is a Argentinian player aged 36, playing as AM (RC), ST (C) for Inter Miami (Major League Soccer). Preferred foot: Left. Playing style: Creative. Status: Available. Transfer value: not for sale. Wage: EUR1,591,000/month. Key attributes: determination 20, dribbling 20, flair 20, technique 20, vision 20, first touch 19, passing 19, balance 18.


In [21]:
# -- Profile sanity check: spot-check 3 random players -----------------------
import random
random.seed(42)
for idx in random.sample(range(len(df)), 3):
    row = df.iloc[idx]
    print(f"--- {row['Player_Name']} ---")
    print(row["profile"])
    print()

--- Millen Burnett ---
Millen Burnett is a English player aged 17, playing as AM (RL) for Hartlepool (Vanarama National League). Preferred foot: Right. Playing style: Physical. Status: Youth. Transfer value: EUR6,000. Wage: EUR250/month. Key attributes: natural fitness 13, acceleration 12, determination 12, flair 12, pace 12, agility 11, first touch 10, stamina 10.

--- Nizamettin Çalışkan ---
Nizamettin Çalışkan is a Turkish player aged 36, playing as DM, M (C) for Bucaspor 1928 (Turkish 2. League White Group). Preferred foot: Right. Playing style: Intelligent. Status: Available. Transfer value: EUR40,000. Wage: EUR3,800/month. Key attributes: leadership 18, composure 16, decisions 15, natural fitness 15, penalties 15, teamwork 15, technique 15, corners 14.

--- Lyle Foster ---
Lyle Foster is a South African player aged 22, playing as AM (L), ST (C) for Burnley (English Premier Division). Preferred foot: Right. Playing style: Physical. Status: Unavailable. Transfer value: EUR33,000,00

---
## Section 6 -- Query Understanding / Constraint Parsing

A **transparent, rule-based parser** extracts structured constraints from the
natural-language query. Unmatched semantic phrases are kept as-is for vector retrieval.
The parser does not need to be perfect -- it must be honest about what it could not extract.

| Constraint type | Example fragment | Parsed as |
|---|---|---|
| Age upper bound | "under 23" | `age_max=22` |
| Age lower bound | "over 25" | `age_min=25` |
| Age range | "between 20 and 25" | `age_min=20, age_max=25` |
| Value upper | "valued under EUR10M" | `value_max=10_000_000` |
| Wage upper | "wage under EUR5000/month" | `wage_max=5000` |
| Position | "centre-back", "striker" | `position_kws=[...]` |
| Foot | "left-footed" | `foot="Left"` |
| Nationality | "Brazilian" | `nationality="Brazilian"` |
| Attribute | "passing > 15" | `attrs={Pas: (">", 15)}` |
| Style | "creative" | `style="Creative"` |
| Semantic | "high-pressing midfielder" | -> vector retrieval |

In [ ]:
# -- Robust EUR amount parser -------------------------------------------------
def parse_eur_amount(s):
    """Parse informal money strings: '10M', '10 million', '5 mil', '500k', '1.5B'."""
    if s is None:
        return None
    s = str(s).lower().strip()
    s = re.sub(r'(?:eur|€|\$|usd|£)', '', s)
    s = s.replace(',', '').strip()
    s = re.sub(r'\bmillion\b|\bmil\b|\bmils\b', 'M', s)
    s = re.sub(r'\bthousand\b', 'K', s, flags=re.IGNORECASE)
    s = re.sub(r'\bbillion\b', 'B', s)
    s = s.strip()
    m = re.match(r'^([0-9]+(?:\.[0-9]+)?)\s*([KMBkmb]?)$', s)
    if m:
        num    = float(m.group(1))
        suffix = m.group(2).upper()
        if suffix == 'K':   num *= 1_000
        elif suffix == 'M': num *= 1_000_000
        elif suffix == 'B': num *= 1_000_000_000
        return num
    return None


def _parse_groups(g1, g2):
    """Combine regex group(1)=number, group(2)=scale-suffix and parse."""
    if g1 is None:
        return None
    return parse_eur_amount((g1 or '') + (' ' + g2 if g2 else ''))


_NUMSFX = r'(?:EUR|€|\$)?\s*([0-9]+(?:\.[0-9]+)?)\s*(million|mil\b|M\b|K\b|thousand\b|B\b|billion\b)?'


# -- Adjective / style attribute synonym map ----------------------------------
_ATTR_SYNONYMS = {
    r'\bfast\b':                    ('Pac', '>=', 14),
    r'\bpacy\b':                    ('Pac', '>=', 14),
    r'\bquick\b':                   ('Pac', '>=', 14),
    r'\belectric\b':                ('Pac', '>=', 15),
    r'\bexplosive\b':               ('Pac', '>=', 15),
    r'\bslow\b':                    ('Pac', '<=', 10),
    r'\bstrong\b':                  ('Str', '>=', 14),
    r'\bpowerful\b':                ('Str', '>=', 14),
    r'\bphysical\b':                ('Str', '>=', 13),
    r'\bbulky\b':                   ('Str', '>=', 14),
    r'\bmuscular\b':                ('Str', '>=', 14),
    r'\bdominant\b':                ('Str', '>=', 14),
    r'\baerial\b':                  ('Hea', '>=', 14),
    r'\bgood in the air\b':         ('Hea', '>=', 14),
    r'\bheader\b':                  ('Hea', '>=', 14),
    r'\bclinical\b':                ('Fin', '>=', 15),
    r'\blethal\b':                  ('Fin', '>=', 15),
    r'\bsharp\b':                   ('Fin', '>=', 14),
    r'\bgoal[- ]?scorer\b':         ('Fin', '>=', 14),
    r'\btireless\b':                ('Sta', '>=', 15),
    r'\bhigh work[- ]?rate\b':      ('Wor', '>=', 15),
    r'\bfit\b':                     ('Sta', '>=', 14),
    r'\benduring\b':                ('Sta', '>=', 14),
    r'\btricky\b':                  ('Dri', '>=', 14),
    r'\bagile\b':                   ('Dri', '>=', 14),
    r'\bskillful\b':                ('Dri', '>=', 14),
    r'\bskilled\b':                 ('Dri', '>=', 14),
    r'\bsilky\b':                   ('Dri', '>=', 14),
    r'\bcreative\b':                ('Vis', '>=', 14),
    r'\bvisionary\b':               ('Vis', '>=', 15),
    r'\bplaymaker\b':               ('Vis', '>=', 14),
    r'\bpasser\b':                  ('Pas', '>=', 14),
    r'\blong[- ]?passer\b':         ('Lon', '>=', 14),
    r'\btackler\b':                 ('Tck', '>=', 14),
    r'\btenacious\b':               ('Tck', '>=', 14),
    r'\bsolid\b':                   ('Tck', '>=', 13),
    r'\breliable\b':                ('Tck', '>=', 13),
    r'\bshot[- ]?stopper\b':        ('Ref', '>=', 14),
    r'\bcalm\b':                    ('Cmp', '>=', 14),
    r'\bcomposed\b':                ('Cmp', '>=', 14),
    r'\bleader\b':                  ('Ldr', '>=', 14),
    r'\bleadership\b':              ('Ldr', '>=', 14),
    r'\bintelligent\b':             ('Dec', '>=', 14),
    r'\bsmart\b':                   ('Dec', '>=', 14),
    r'\bclever\b':                  ('Dec', '>=', 14),
    r'\bpressing\b':                ('Wor', '>=', 14),
    r'\bhigh[- ]?press(?:ing)?\b':  ('Wor', '>=', 14),
    r'\bwork[- ]?rate\b':           ('Wor', '>=', 14),
    r'\benerget(?:ic|ics)\b':       ('Sta', '>=', 14),
    r'\bdynamic\b':                 ('Wor', '>=', 14),
    r'\bengine\b':                  ('Sta', '>=', 15),
    r'\bindustrious\b':             ('Wor', '>=', 15),
    r'\bintelligent runner\b':      ('OtB', '>=', 14),
    r'\boff the ball\b':            ('OtB', '>=', 14),
    r'\bgood movement\b':           ('OtB', '>=', 14),
    r'\bpositionally aware\b':      ('Pos', '>=', 14),
    r'\bdetermined\b':              ('Det', '>=', 14),
    r'\bbrave\b':                   ('Bra', '>=', 14),
    r'\bbalanced\b':                ('Bal', '>=', 14),
    r'\bstamina\b':                 ('Sta', '>=', 14),
    r'\bpace\b':                    ('Pac', '>=', 14),
    r'\btechnique\b':               ('Tec', '>=', 14),
    r'\bvision\b':                  ('Vis', '>=', 14),
    r'\bdecision\b':                ('Dec', '>=', 14),
    r'\banticipation\b':            ('Ant', '>=', 14),
    r'\bcomposure\b':               ('Cmp', '>=', 14),
    r'\baggressive\b':              ('Agg', '>=', 14),
    r'\bworkrate\b':                ('Wor', '>=', 14),
}

_AGE_SYNONYMS = {
    r'\bteenager\b':    (None, 19),
    r'\bteen\b':        (None, 19),
    r'\byoungster\b':   (None, 21),
    r'\byoung\b':       (None, 23),
    r'\bprospect\b':    (None, 23),
    r'\bveteran\b':     (30, None),
    r'\bexperienced\b': (28, None),
    r'\bsenior\b':      (28, None),
    r'\bmature\b':      (27, None),
}

_NAT_MAP = {
    'south korean': 'South Korean',
    'costa rican':  'Costa Rican',
    'argentine':    'Argentinian',
    'argentinian':  'Argentinian',
    'brazilian':    'Brazilian',
    'french':       'French',
    'spanish':      'Spanish',
    'english':      'English',
    'german':       'German',
    'italian':      'Italian',
    'portuguese':   'Portuguese',
    'dutch':        'Dutch',
    'belgian':      'Belgian',
    'polish':       'Polish',
    'norwegian':    'Norwegian',
    'egyptian':     'Egyptian',
    'senegalese':   'Senegalese',
    'turkish':      'Turkish',
    'scottish':     'Scottish',
    'irish':        'Irish',
    'welsh':        'Welsh',
    'croatian':     'Croatian',
    'uruguayan':    'Uruguayan',
    'colombian':    'Colombian',
    'moroccan':     'Moroccan',
    'nigerian':     'Nigerian',
    'japanese':     'Japanese',
    'korean':       'South Korean',
    'australian':   'Australian',
    'american':     'American',
    'mexican':      'Mexican',
    'danish':       'Danish',
    'swedish':      'Swedish',
    'swiss':        'Swiss',
    'greek':        'Greek',
    'austrian':     'Austrian',
    'czech':        'Czech',
    'romanian':     'Romanian',
    'ukrainian':    'Ukrainian',
    'ghanaian':     'Ghanaian',
    'algerian':     'Algerian',
    'chilean':      'Chilean',
    'peruvian':     'Peruvian',
    'ecuadorian':   'Ecuadorian',
    'venezuelan':   'Venezuelan',
    'jamaican':     'Jamaican',
    'iranian':      'Iranian',
    'chinese':      'Chinese',
    'thai':         'Thai',
    'indonesian':   'Indonesian',
    'serbian':      'Serbian',
    'hungarian':    'Hungarian',
    'bulgarian':    'Bulgarian',
    'tunisian':     'Tunisian',
    'ivorian':      'Ivorian',
    'cameroonian':  'Cameroonian',
}


# -- Main query parser --------------------------------------------------------
def parse_query(query):
    """
    Robust natural-language scouting query parser.
    Returns dict: age_min, age_max, value_min, value_max, wage_min, wage_max,
    position_kws, foot, nationality, style, attrs, semantic_query, raw_query,
    parsed_tokens, unparsed_notes
    """
    q = query.lower()
    result = {
        'age_min': None, 'age_max': None,
        'value_min': None, 'value_max': None,
        'wage_min': None, 'wage_max': None,
        'position_kws': [],
        'foot': None, 'nationality': None, 'style': None,
        'attrs': {},
        'semantic_query': query, 'raw_query': query,
        'parsed_tokens': [], 'unparsed_notes': [],
    }

    def record(key, val, span):
        result[key] = val
        result['parsed_tokens'].append(f'{key}={val}  [from: "{span}"]')

    def record_attr(col, op, val, span):
        result['attrs'][col] = (op, val)
        result['parsed_tokens'].append(f'attr {col}{op}{val}  [from: "{span}"]')

    # 1. AGE
    for m in re.finditer(r'\b(?:under|younger than|u-?)\s*(\d+)\b', q):
        record('age_max', int(m.group(1)) - 1, m.group(0))
    for m in re.finditer(r'\b(?:aged?|age)\b\s+(?:under|below|less than)\s+(\d+)', q):
        record('age_max', int(m.group(1)) - 1, m.group(0))
    for m in re.finditer(r'\b(?:older than|aged? over|aged? above)\s*(\d+)', q):
        record('age_min', int(m.group(1)), m.group(0))
    for m in re.finditer(r'\bno\s+older\s+than\s*(\d+)', q):
        record('age_max', int(m.group(1)), m.group(0))
    for m in re.finditer(r'\b(?:between|aged?)\s+(\d+)\s+(?:and|to|-)\s+(\d+)', q):
        record('age_min', int(m.group(1)), m.group(0))
        record('age_max', int(m.group(2)), m.group(0))
    for pattern, (mn, mx) in _AGE_SYNONYMS.items():
        if re.search(pattern, q):
            if mn is not None and result['age_min'] is None:
                record('age_min', mn, pattern.replace(r'\b', ''))
            if mx is not None and result['age_max'] is None:
                record('age_max', mx, pattern.replace(r'\b', ''))

    # 2. TRANSFER VALUE
    _vmax_pat = (
        r'(?:'
        r'(?:budget|price tag|asking price|transfer fee)'
        r'\s*(?:of|is|around|~|approx(?:imately)?|about)?'
        r'\s*(?:around|~|up to|approx(?:imately)?|about|max(?:imum)?|no more than)?'
        r'|up to|max(?:imum)?|no more than|at most'
        r'|costs?\s+(?:less than|under|below|around|about)'
        r'|worth\s+(?:less than|under|below)'
        r'|valued?\s+(?:at most|under|below|less than)'
        r'|market value\s+(?:under|below|less than)'
        r'|priced?\s+(?:under|below|at most)'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_vmax_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['value_max'] is None:
            record('value_max', v, m.group(0).strip())

    _vmin_pat = (
        r'(?:'
        r'valued?\s+(?:over|above|more than)'
        r'|worth\s+(?:more than|over|above)'
        r'|(?:minimum|min)\s+(?:value|price|fee)?\s*(?:of)?'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_vmin_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['value_min'] is None:
            record('value_min', v, m.group(0).strip())

    # 3. WAGE
    _wmax_pat = (
        r'(?:'
        r'wages?\s+(?:under|below|less than|of\s+(?:under|below|at most))'
        r'|(?:earning|paid|earns?)\s+(?:less than|under|below|at most|no more than)'
        r'|(?:weekly\s+)?(?:salary|wage)\s+(?:under|below|less than|of\s+(?:up to|at most))'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_wmax_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['wage_max'] is None:
            record('wage_max', v, m.group(0).strip())

    _wmin_pat = (
        r'(?:'
        r'wages?\s+(?:over|above|more than|at least)'
        r'|(?:earning|earns?|paid)\s+(?:more than|over|above|at least)'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_wmin_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['wage_min'] is None:
            record('wage_min', v, m.group(0).strip())

    # 4. FOOT
    if re.search(r'\bleft[- ]?foot(?:ed)?\b', q):
        record('foot', 'Left', 'left-footed')
    if re.search(r'\bright[- ]?foot(?:ed)?\b', q):
        record('foot', 'Right', 'right-footed')

    # 5. POSITION
    for kw in sorted(POSITION_KEYWORDS, key=len, reverse=True):
        if re.search(r'\b' + re.escape(kw) + r'\b', q):
            if not result['position_kws']:
                result['position_kws'] = POSITION_KEYWORDS[kw]
                result['parsed_tokens'].append(
                    f'position_kws={result["position_kws"]}  [from: "{kw}"]'
                )

    # 6. EXPLICIT attribute constraints  "pace >= 14"
    for attr_kw, col in sorted(ATTR_KEYWORDS.items(), key=lambda x: -len(x[0])):
        for m in re.finditer(
            r'\b' + re.escape(attr_kw) + r'\b\s*'
            r'(>=?|<=?|>|<|=|at least|above|over|below|under)\s*(\d+)', q
        ):
            op_raw = m.group(1).strip()
            threshold = int(m.group(2))
            op = {'at least': '>=', 'above': '>', 'over': '>',
                  'below': '<', 'under': '<'}.get(op_raw, op_raw)
            result['attrs'][col] = (op, threshold)
            result['parsed_tokens'].append(
                f'attr {col}{op}{threshold}  [from: "{m.group(0)}"]'
            )

    # 7. ADJECTIVE synonyms
    for pattern, (col, op, threshold) in _ATTR_SYNONYMS.items():
        if re.search(pattern, q):
            if col not in result['attrs']:
                record_attr(col, op, threshold, pattern.replace(r'\b', ''))

    # 8. NATIONALITY
    _nat_pat = (r'\b(' +
                '|'.join(re.escape(k) for k in sorted(_NAT_MAP, key=len, reverse=True)) +
                r')\b')
    for m in re.finditer(_nat_pat, q):
        result['nationality'] = _NAT_MAP[m.group(1)]
        result['parsed_tokens'].append(
            f'nationality={result["nationality"]}  [from: "{m.group(0)}"]'
        )
        break

    # 9. Semantic residual
    semantic = query
    _strip = [
        r'\b(?:under|younger than|u-?)\s*\d+\b',
        r'(?:aged?|age)\s+(?:under|below|over|above|less than|between \d+ and)\s*\d+',
        r'\bbetween \d+ and \d+\b',
        r'\bno\s+older\s+than\s*\d+',
        r'\bolder than \d+\b',
        (r'(?:budget|price tag|up to|max(?:imum)?|no more than|at most|costs?'
         r'|worth|valued?|market value|priced?)\s*(?:around|~|about|approx(?:imately)?)?'
         r'\s*(?:EUR|€|\$)?\s*[\d,.]+\s*(?:million|mil|M|K|thousand|B|billion)?'),
        (r'(?:wages?|salary|earning|paid)\s*(?:under|below|over|above|less than'
         r'|more than|at least|at most|no more than)\s*(?:EUR|€|\$)?'
         r'\s*[\d,.]+\s*(?:million|mil|M|K|thousand|B|billion)?'),
        r'\bleft[- ]?foot(?:ed)?\b',
        r'\bright[- ]?foot(?:ed)?\b',
    ]
    for pat in _strip:
        semantic = re.sub(pat, ' ', semantic, flags=re.IGNORECASE)
    semantic = re.sub(r'\s+', ' ', semantic).strip(' ,.-')
    result['semantic_query'] = semantic if semantic else query
    return result


# -- Sanity tests -------------------------------------------------------------
_tests = [
    "Find a left-footed centre-back under 23 with passing > 15 and tackling >= 14, valued under EUR10M",
    "get me a fast striker, budget around 10 million, under 25",
    "powerful defensive midfielder, max budget 8M, experienced",
    "up to 5 mil, pacy winger under 21",
    "veteran goalkeeper, wages under 50K",
    "creative attacking midfielder, teenager, costs less than 3 million",
    "strong aerial centre back, South Korean, budget around 15M",
    "clinical striker earning less than 80K per week",
    "Argentine winger under 22, valued at most 12M",
    "tireless box-to-box midfielder, no older than 27, wage below 100K",
    "High-pressing box-to-box midfielder with excellent stamina and work rate",
]
print("=" * 65)
print("Robust parser -- sanity tests")
print("=" * 65)
for tq in _tests:
    p = parse_query(tq)
    print(f"\nQ: {tq}")
    for tok in p['parsed_tokens']:
        print(f"  [OK] {tok}")
print("=" * 65)
print("Parser ready.")


# ---------------------------------------------------------------------------
# LLM Query Rewriter  (requires LLM_BACKEND = "local" and model loaded)
# ---------------------------------------------------------------------------
# Architecture:
#   informal query -> [Qwen rewriter] -> structured query -> [parse_query]
#
# Falls back to parse_query on original query if LLM is not loaded.
# ---------------------------------------------------------------------------

_REWRITE_SYSTEM = (
    "You are a query rewriter for a football scouting system.\n"
    "Rewrite the user's informal scouting query into a clean structured form.\n\n"
    "RULES:\n"
    "1. Keep all position names (striker, midfielder, centre-back, winger, goalkeeper).\n"
    "2. Convert adjectives into explicit FM24 attribute constraints:\n"
    "   pace, stamina, strength, work rate, aggression, tackling, passing, vision,\n"
    "   dribbling, finishing, heading, off the ball, positioning, determination,\n"
    "   composure, anticipation, decisions, technique, flair, balance, bravery.\n"
    "3. Use format: attribute >= value  (14 = good/high, 15 = excellent).\n"
    "4. Keep age constraints: under 23, older than 28, teenager, veteran.\n"
    "5. Keep budget/value: valued under 10M EUR, budget around 5M.\n"
    "6. Keep wages: wages under 50K.\n"
    "7. Keep nationality and foot preference.\n"
    "8. Output ONLY the rewritten query, nothing else.\n\n"
    "EXAMPLES:\n"
    'Input:  "high pressing tireless box-to-box midfielder"\n'
    'Output: "box-to-box midfielder, work rate >= 14, stamina >= 15, aggression >= 13"\n\n'
    'Input:  "clinical poacher, cheap, teenager"\n'
    'Output: "striker, finishing >= 15, off the ball >= 14, under 19, valued under 5M EUR"\n\n'
    'Input:  "pacy winger who can beat his man, under 22"\n'
    'Output: "winger, pace >= 14, dribbling >= 14, under 22"\n'
)


def rewrite_query(query):
    """
    Use the loaded local LLM to rewrite an informal query into a structured
    form that parse_query() can reliably handle.
    Returns (rewritten_str, was_rewritten_bool).
    Falls back to (original_query, False) if model is not loaded.
    """
    try:
        if '_local_model' not in globals() or _local_model is None:
            return query, False

        messages = [
            {"role": "system", "content": _REWRITE_SYSTEM},
            {"role": "user",   "content": 'Input:  "' + query + '"\nOutput:'},
        ]
        text = _local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs  = _local_tokenizer([text], return_tensors="pt").to(_local_model.device)
        out_ids = _local_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.1,
            do_sample=False,
            pad_token_id=_local_tokenizer.eos_token_id,
        )
        new_ids   = out_ids[:, inputs["input_ids"].shape[-1]:]
        rewritten = _local_tokenizer.decode(new_ids[0], skip_special_tokens=True).strip()
        rewritten = re.sub(r'^(?:output|rewritten|query)\s*:\s*', '', rewritten,
                           flags=re.IGNORECASE).strip('"\'')
        return rewritten, True

    except Exception as e:
        print(f"  [rewrite_query] fallback: {e}")
        return query, False


def smart_parse(query, verbose=True):
    """
    Full pipeline:
      1. Rewrite query with LLM if model is loaded.
      2. Run parse_query() on the rewritten (or original) query.
    Returns the same dict as parse_query(), plus 'rewritten_query' key.
    """
    rewritten, was_rewritten = rewrite_query(query)
    if was_rewritten and verbose:
        print(f"  [LLM rewrite] '{query}'")
        print(f"             -> '{rewritten}'")
    parsed = parse_query(rewritten if was_rewritten else query)
    parsed['rewritten_query'] = rewritten if was_rewritten else None
    parsed['raw_query']       = query
    return parsed


In [ ]:
# -- Robust EUR amount parser -------------------------------------------------
def parse_eur_amount(s):
    """
    Parse informal money strings to float.
    Handles: '10M', '10 million', '5 mil', '500k', '500 thousand',
             '1.5B', 'EUR10M', plain numbers.
    """
    if s is None:
        return None
    s = str(s).lower().strip()
    s = re.sub(r'(?:eur|€|\$|usd|£)', '', s)
    s = s.replace(',', '').strip()
    s = re.sub(r'\bmillion\b|\bmil\b|\bmils\b', 'M', s)
    s = re.sub(r'\bthousand\b', 'K', s, flags=re.IGNORECASE)
    s = re.sub(r'\bbillion\b', 'B', s)
    s = s.strip()
    m = re.match(r'^([0-9]+(?:\.[0-9]+)?)\s*([KMBkmb]?)$', s)
    if m:
        num    = float(m.group(1))
        suffix = m.group(2).upper()
        if suffix == 'K':   num *= 1_000
        elif suffix == 'M': num *= 1_000_000
        elif suffix == 'B': num *= 1_000_000_000
        return num
    return None


# Helper: parse number + suffix captured in two separate regex groups
def _parse_groups(g1, g2):
    """Combine regex group(1)=number, group(2)=scale-suffix and parse."""
    if g1 is None:
        return None
    return parse_eur_amount((g1 or "") + (" " + g2 if g2 else ""))


# Shared number + suffix pattern appended to constraint prefixes
_NUMSFX = r'(?:EUR|€|\$)?\s*([0-9]+(?:\.[0-9]+)?)\s*(million|mil\b|M\b|K\b|thousand\b|B\b|billion\b)?'


# -- Adjective / style attribute synonym map ----------------------------------
_ATTR_SYNONYMS = {
    r'\bfast\b':                ('Pac', '>=', 14),
    r'\bpacy\b':                ('Pac', '>=', 14),
    r'\bquick\b':               ('Pac', '>=', 14),
    r'\belectric\b':            ('Pac', '>=', 15),
    r'\bexplosive\b':           ('Pac', '>=', 15),
    r'\bslow\b':                ('Pac', '<=', 10),
    r'\bstrong\b':              ('Str', '>=', 14),
    r'\bpowerful\b':            ('Str', '>=', 14),
    r'\bphysical\b':            ('Str', '>=', 13),
    r'\bbulky\b':               ('Str', '>=', 14),
    r'\bmuscular\b':            ('Str', '>=', 14),
    r'\bdominant\b':            ('Str', '>=', 14),
    r'\baerial\b':              ('Hea', '>=', 14),
    r'\bgood in the air\b':     ('Hea', '>=', 14),
    r'\bheader\b':              ('Hea', '>=', 14),
    r'\bclinical\b':            ('Fin', '>=', 15),
    r'\blethal\b':              ('Fin', '>=', 15),
    r'\bsharp\b':               ('Fin', '>=', 14),
    r'\bgoal[- ]?scorer\b':     ('Fin', '>=', 14),
    r'\btireless\b':            ('Sta', '>=', 15),
    r'\bhigh work[- ]?rate\b':  ('Wor', '>=', 15),
    r'\bfit\b':                 ('Sta', '>=', 14),
    r'\benduring\b':            ('Sta', '>=', 14),
    r'\btricky\b':              ('Dri', '>=', 14),
    r'\bagile\b':               ('Dri', '>=', 14),
    r'\bskillful\b':            ('Dri', '>=', 14),
    r'\bskilled\b':             ('Dri', '>=', 14),
    r'\bsilky\b':               ('Dri', '>=', 14),
    r'\bcreative\b':            ('Vis', '>=', 14),
    r'\bvisionary\b':           ('Vis', '>=', 15),
    r'\bplaymaker\b':           ('Vis', '>=', 14),
    r'\bpasser\b':              ('Pas', '>=', 14),
    r'\blong[- ]?passer\b':     ('Lon', '>=', 14),
    r'\btackler\b':             ('Tck', '>=', 14),
    r'\btenacious\b':           ('Tck', '>=', 14),
    r'\bsolid\b':               ('Tck', '>=', 13),
    r'\breliable\b':            ('Tck', '>=', 13),
    r'\bshot[- ]?stopper\b':    ('Ref', '>=', 14),
    r'\bcalm\b':                ('Cmp', '>=', 14),
    r'\bcomposed\b':            ('Cmp', '>=', 14),
    r'\bleader\b':              ('Ldr', '>=', 14),
    r'\bleadership\b':          ('Ldr', '>=', 14),
    r'\bintelligent\b':         ('Dec', '>=', 14),
    r'\bsmart\b':               ('Dec', '>=', 14),
    r'\bclever\b':              ('Dec', '>=', 14),
    # Pressing / work rate
    r'\bpressing\b':            ('Wor', '>=', 14),
    r'\bhigh[- ]?press(?:ing)?\b': ('Wor', '>=', 14),
    r'\bwork[- ]?rate\b':        ('Wor', '>=', 14),
    r'\benerget(?:ic|ics)\b':    ('Sta', '>=', 14),
    r'\bdynamic\b':              ('Wor', '>=', 14),
    r'\bengine\b':               ('Sta', '>=', 15),
    r'\bindustrious\b':          ('Wor', '>=', 15),
    r'\bboxer\b':                ('Sta', '>=', 14),
    # Anticipation / positioning
    r'\bintelligent runner\b':   ('OtB', '>=', 14),
    r'\boff the ball\b':         ('OtB', '>=', 14),
    r'\bgood movement\b':        ('OtB', '>=', 14),
    r'\bpositionally aware\b':   ('Pos', '>=', 14),
    r'\bdetermined\b':           ('Det', '>=', 14),
    r'\bbrave\b':                ('Bra', '>=', 14),
    r'\bbalanced\b':             ('Bal', '>=', 14),
    # Standalone FM attribute words (without explicit threshold)
    r'\bstamina\b':               ('Sta', '>=', 14),
    r'\bpace\b':                  ('Pac', '>=', 14),
    r'\btechnique\b':             ('Tec', '>=', 14),
    r'\bvision\b':                ('Vis', '>=', 14),
    r'\bdecision\b':              ('Dec', '>=', 14),
    r'\banticipation\b':          ('Ant', '>=', 14),
    r'\bcomposure\b':             ('Cmp', '>=', 14),
    r'\baggressive\b':            ('Agg', '>=', 14),
    r'\bworkrate\b':              ('Wor', '>=', 14),
}

# -- Age synonym map ----------------------------------------------------------
_AGE_SYNONYMS = {
    r'\bteenager\b':    (None, 19),
    r'\bteen\b':        (None, 19),
    r'\byoungster\b':   (None, 21),
    r'\byoung\b':       (None, 23),
    r'\bprospect\b':    (None, 23),
    r'\bveteran\b':     (30, None),
    r'\bexperienced\b': (28, None),
    r'\bsenior\b':      (28, None),
    r'\bmature\b':      (27, None),
}

# -- Nationality normalisation map -------------------------------------------
_NAT_MAP = {
    'south korean': 'South Korean',
    'costa rican':  'Costa Rican',
    'argentine':    'Argentinian',
    'argentinian':  'Argentinian',
    'brazilian':    'Brazilian',
    'french':       'French',
    'spanish':      'Spanish',
    'english':      'English',
    'german':       'German',
    'italian':      'Italian',
    'portuguese':   'Portuguese',
    'dutch':        'Dutch',
    'belgian':      'Belgian',
    'polish':       'Polish',
    'norwegian':    'Norwegian',
    'egyptian':     'Egyptian',
    'senegalese':   'Senegalese',
    'turkish':      'Turkish',
    'scottish':     'Scottish',
    'irish':        'Irish',
    'welsh':        'Welsh',
    'croatian':     'Croatian',
    'uruguayan':    'Uruguayan',
    'colombian':    'Colombian',
    'moroccan':     'Moroccan',
    'nigerian':     'Nigerian',
    'japanese':     'Japanese',
    'korean':       'South Korean',
    'australian':   'Australian',
    'american':     'American',
    'mexican':      'Mexican',
    'danish':       'Danish',
    'swedish':      'Swedish',
    'swiss':        'Swiss',
    'greek':        'Greek',
    'austrian':     'Austrian',
    'czech':        'Czech',
    'romanian':     'Romanian',
    'ukrainian':    'Ukrainian',
    'ghanaian':     'Ghanaian',
    'algerian':     'Algerian',
    'chilean':      'Chilean',
    'peruvian':     'Peruvian',
    'ecuadorian':   'Ecuadorian',
    'venezuelan':   'Venezuelan',
    'jamaican':     'Jamaican',
    'iranian':      'Iranian',
    'chinese':      'Chinese',
    'thai':         'Thai',
    'indonesian':   'Indonesian',
    'serbian':      'Serbian',
    'hungarian':    'Hungarian',
    'bulgarian':    'Bulgarian',
    'tunisian':     'Tunisian',
    'ivorian':      'Ivorian',
    'cameroonian':  'Cameroonian',
}


# -- Main query parser (robust version) --------------------------------------
def parse_query(query):
    """
    Robust natural-language scouting query parser.

    Handles informal phrasing like:
      "budget around 10M", "fast striker", "veteran", "powerful",
      "tireless box-to-box midfielder", "Argentine winger under 22"

    Returns dict with keys:
      age_min, age_max, value_min, value_max, wage_min, wage_max,
      position_kws, foot, nationality, style, attrs,
      semantic_query, raw_query, parsed_tokens, unparsed_notes
    """
    q = query.lower()
    result = {
        'age_min':      None, 'age_max':      None,
        'value_min':    None, 'value_max':    None,
        'wage_min':     None, 'wage_max':     None,
        'position_kws': [],
        'foot':         None,
        'nationality':  None,
        'style':        None,
        'attrs':        {},
        'semantic_query': query,
        'raw_query':    query,
        'parsed_tokens':  [],
        'unparsed_notes': [],
    }

    def record(key, val, span):
        result[key] = val
        result['parsed_tokens'].append(f'{key}={val}  [from: "{span}"]')

    def record_attr(col, op, val, span):
        result['attrs'][col] = (op, val)
        result['parsed_tokens'].append(f'attr {col}{op}{val}  [from: "{span}"]')

    # ------------------------------------------------------------------ #
    # 1. AGE constraints                                                   #
    # ------------------------------------------------------------------ #
    # "under 23", "u23", "u-23", "younger than 23"
    for m in re.finditer(r'\b(?:under|younger than|u-?)\s*(\d+)\b', q):
        record('age_max', int(m.group(1)) - 1, m.group(0))
    for m in re.finditer(r'\b(?:aged?|age)\b\s+(?:under|below|less than)\s+(\d+)', q):
        record('age_max', int(m.group(1)) - 1, m.group(0))

    # "older than 28", "aged over 30"
    for m in re.finditer(r'\b(?:older than|aged? over|aged? above)\s*(\d+)', q):
        record('age_min', int(m.group(1)), m.group(0))
    # "no older than 27" -> age_max
    for m in re.finditer(r'\bno\s+older\s+than\s*(\d+)', q):
        record('age_max', int(m.group(1)), m.group(0))

    # "between 20 and 26", "aged 20-26"
    for m in re.finditer(r'\b(?:between|aged?)\s+(\d+)\s+(?:and|to|-)\s+(\d+)', q):
        record('age_min', int(m.group(1)), m.group(0))
        record('age_max', int(m.group(2)), m.group(0))

    # Age synonym words (only if explicit age not already set)
    for pattern, (mn, mx) in _AGE_SYNONYMS.items():
        if re.search(pattern, q):
            if mn is not None and result['age_min'] is None:
                record('age_min', mn, pattern.replace(r'\b', ''))
            if mx is not None and result['age_max'] is None:
                record('age_max', mx, pattern.replace(r'\b', ''))

    # ------------------------------------------------------------------ #
    # 2. TRANSFER VALUE constraints                                        #
    # ------------------------------------------------------------------ #
    # value_max: budget / up to / max / costs / worth (less than) / valued under
    # All alternatives are inside (?:...) so the | stays scoped to the prefix group
    _vmax_pat = (
        r'(?:'
        r'(?:budget|price tag|asking price|transfer fee)'
        r'\s*(?:of|is|around|~|approx(?:imately)?|about)?'
        r'\s*(?:around|~|up to|approx(?:imately)?|about|max(?:imum)?|no more than)?'
        r'|up to|max(?:imum)?|no more than|at most'
        r'|costs?\s+(?:less than|under|below|around|about)'
        r'|worth\s+(?:less than|under|below)'
        r'|valued?\s+(?:at most|under|below|less than)'
        r'|market value\s+(?:under|below|less than)'
        r'|priced?\s+(?:under|below|at most)'
        r'|affordable at'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_vmax_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['value_max'] is None:
            record('value_max', v, m.group(0).strip())

    # value_min: worth at least / valued over / minimum value
    _vmin_pat = (
        r'(?:'
        r'valued?\s+(?:over|above|more than)'
        r'|worth\s+(?:more than|over|above)'
        r'|(?:minimum|min)\s+(?:value|price|fee)?\s*(?:of)?'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_vmin_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['value_min'] is None:
            record('value_min', v, m.group(0).strip())

    # ------------------------------------------------------------------ #
    # 3. WAGE constraints                                                  #
    # ------------------------------------------------------------------ #
    _wmax_pat = (
        r'(?:'
        r'wages?\s+(?:under|below|less than|of\s+(?:under|below|at most))'
        r'|(?:earning|paid|earns?)\s+(?:less than|under|below|at most|no more than)'
        r'|(?:weekly\s+)?(?:salary|wage)\s+(?:under|below|less than|of\s+(?:up to|at most))'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_wmax_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['wage_max'] is None:
            record('wage_max', v, m.group(0).strip())

    _wmin_pat = (
        r'(?:'
        r'wages?\s+(?:over|above|more than|at least)'
        r'|(?:earning|earns?|paid)\s+(?:more than|over|above|at least)'
        r')\s*' + _NUMSFX
    )
    for m in re.finditer(_wmin_pat, q, re.IGNORECASE):
        v = _parse_groups(m.group(1), m.group(2))
        if v and result['wage_min'] is None:
            record('wage_min', v, m.group(0).strip())

    # ------------------------------------------------------------------ #
    # 4. PREFERRED FOOT                                                    #
    # ------------------------------------------------------------------ #
    if re.search(r'\bleft[- ]?foot(?:ed)?\b', q):
        record('foot', 'Left', 'left-footed')
    if re.search(r'\bright[- ]?foot(?:ed)?\b', q):
        record('foot', 'Right', 'right-footed')

    # ------------------------------------------------------------------ #
    # 5. POSITION keywords (longest match first)                           #
    # ------------------------------------------------------------------ #
    for kw in sorted(POSITION_KEYWORDS, key=len, reverse=True):
        if re.search(r'\b' + re.escape(kw) + r'\b', q):
            if not result['position_kws']:
                result['position_kws'] = POSITION_KEYWORDS[kw]
                result['parsed_tokens'].append(
                    f'position_kws={result["position_kws"]}  [from: "{kw}"]'
                )

    # ------------------------------------------------------------------ #
    # 6. EXPLICIT attribute constraints  e.g. "pace >= 14", "Pac > 13"    #
    # ------------------------------------------------------------------ #
    for attr_kw, col in sorted(ATTR_KEYWORDS.items(), key=lambda x: -len(x[0])):
        for m in re.finditer(
            r'\b' + re.escape(attr_kw) + r'\b\s*'
            r'(>=?|<=?|>|<|=|at least|above|over|below|under)\s*(\d+)', q
        ):
            op_raw    = m.group(1).strip()
            threshold = int(m.group(2))
            op = {'at least': '>=', 'above': '>', 'over': '>',
                  'below': '<', 'under': '<'}.get(op_raw, op_raw)
            result['attrs'][col] = (op, threshold)
            result['parsed_tokens'].append(
                f'attr {col}{op}{threshold}  [from: "{m.group(0)}"]'
            )

    # ------------------------------------------------------------------ #
    # 7. ADJECTIVE synonyms -> implicit attribute constraints              #
    # ------------------------------------------------------------------ #
    for pattern, (col, op, threshold) in _ATTR_SYNONYMS.items():
        if re.search(pattern, q):
            if col not in result['attrs']:   # explicit attr takes priority
                record_attr(col, op, threshold, pattern.replace(r'\b', ''))

    # ------------------------------------------------------------------ #
    # 8. NATIONALITY                                                        #
    # ------------------------------------------------------------------ #
    _nat_pattern = (r'\b(' +
                    '|'.join(re.escape(k) for k in sorted(_NAT_MAP, key=len, reverse=True)) +
                    r')\b')
    for m in re.finditer(_nat_pattern, q):
        result['nationality'] = _NAT_MAP[m.group(1)]
        result['parsed_tokens'].append(
            f'nationality={result["nationality"]}  [from: "{m.group(0)}"]'
        )
        break

    # ------------------------------------------------------------------ #
    # 9. Semantic residual (strip parsed boilerplate from query)           #
    # ------------------------------------------------------------------ #
    semantic = query
    _strip_pats = [
        r'\b(?:under|younger than|u-?)\s*\d+\b',
        r'(?:aged?|age)\s+(?:under|below|over|above|less than|between \d+ and)\s*\d+',
        r'\bbetween \d+ and \d+\b',
        r'\bno\s+older\s+than\s*\d+',
        r'\bolder than \d+\b',
        (r'(?:budget|price tag|up to|max(?:imum)?|no more than|at most|costs?'
         r'|worth|valued?|market value|priced?)\s*(?:around|~|about|approx(?:imately)?)?'
         r'\s*(?:EUR|€|\$)?\s*[\d,.]+\s*(?:million|mil|M|K|thousand|B|billion)?'),
        (r'(?:wages?|salary|earning|paid)\s*(?:under|below|over|above|less than'
         r'|more than|at least|at most|no more than)\s*(?:EUR|€|\$)?'
         r'\s*[\d,.]+\s*(?:million|mil|M|K|thousand|B|billion)?'),
        r'\bleft[- ]?foot(?:ed)?\b',
        r'\bright[- ]?foot(?:ed)?\b',
    ]
    for pat in _strip_pats:
        semantic = re.sub(pat, ' ', semantic, flags=re.IGNORECASE)
    semantic = re.sub(r'\s+', ' ', semantic).strip(' ,.-')
    result['semantic_query'] = semantic if semantic else query

    return result


# -- Sanity tests -------------------------------------------------------
_tests = [
    "Find a left-footed centre-back under 23 with passing > 15 and tackling >= 14, valued under EUR10M",
    "get me a fast striker, budget around 10 million, under 25",
    "powerful defensive midfielder, max budget 8M, experienced",
    "up to 5 mil, pacy winger under 21",
    "veteran goalkeeper, wages under 50K",
    "creative attacking midfielder, teenager, costs less than 3 million",
    "strong aerial centre back, South Korean, budget around 15M",
    "clinical striker earning less than 80K per week",
    "Argentine winger under 22, valued at most 12M",
    "tireless box-to-box midfielder, no older than 27, wage below 100K",
]
print("=" * 65)
print("Robust parser -- sanity tests")
print("=" * 65)
for tq in _tests:
    p = parse_query(tq)
    print(f"\nQ: {tq}")
    for tok in p['parsed_tokens']:
        print(f"  [OK] {tok}")
    print(f"  Semantic: '{p['semantic_query']}'")
print("=" * 65)
print("Parser ready.")


# ---------------------------------------------------------------------------
# LLM-assisted constraint extraction (optional enhancement)
# ---------------------------------------------------------------------------
# When the regex parser misses fuzzy tactical terms ("high pressing",
# "box-to-box engine", "tenacious presser"), this function asks the
# already-loaded local Qwen model to extract structured constraints and
# merges them with the regex result.
# Requires LLM_BACKEND = "local" and the model loaded in the LLM cell.
# ---------------------------------------------------------------------------

_ATTR_JSON_HINT = (
    "Pac=pace, Acc=acceleration, Sta=stamina, Str=strength, Wor=work_rate, "
    "Agg=aggression, Tck=tackling, Mar=marking, Pas=passing, Vis=vision, "
    "Dri=dribbling, Fin=finishing, Hea=heading, OtB=off_the_ball, "
    "Pos=positioning, Det=determination, Ant=anticipation, Dec=decisions, "
    "Cmp=composure, Bal=balance, Bra=bravery, Fla=flair, Lon=long_shots, "
    "Cro=crossing, Agi=agility, Aer=aerial, Ldr=leadership"
)

_LLM_EXTRACT_SYSTEM = (
    "You are a football attribute extractor for a Football Manager 2024 dataset.
"
    "Given a scouting query, output ONLY a valid JSON object with any numeric attribute "
    "constraints implied by the query.
"
    "Use these FM24 short codes: " + _ATTR_JSON_HINT + "
"
    "Format: {"attrs": {"CODE": [">=", value], ...}}
"
    "Use >= for 'good/high' (threshold 14), >= 15 for 'excellent/world-class'.
"
    "Only include attributes that are clearly implied. Output JSON only, no explanation."
)

def llm_extract_constraints(query, parsed):
    """
    Use the loaded local LLM to extract implicit attribute constraints
    that the regex parser missed, then merge into the parsed dict.
    Returns updated parsed dict (original is NOT mutated).
    """
    try:
        # Only works if local model is already loaded
        if '_local_model' not in globals() or _local_model is None:
            return parsed

        import copy, re as _re
        updated = copy.deepcopy(parsed)

        messages = [
            {"role": "system", "content": _LLM_EXTRACT_SYSTEM},
            {"role": "user",   "content": f"Query: {query}"},
        ]
        text = _local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs  = _local_tokenizer([text], return_tensors="pt").to(_local_model.device)
        out_ids = _local_model.generate(
            **inputs, max_new_tokens=200, temperature=0.1, do_sample=False,
            pad_token_id=_local_tokenizer.eos_token_id,
        )
        new_ids = out_ids[:, inputs["input_ids"].shape[-1]:]
        raw = _local_tokenizer.decode(new_ids[0], skip_special_tokens=True).strip()

        # Extract JSON from the response (model may add text around it)
        json_match = _re.search(r'\{.*\}', raw, _re.DOTALL)
        if not json_match:
            return updated
        extracted = json.loads(json_match.group(0))

        merged = 0
        for col, val in extracted.get("attrs", {}).items():
            if col not in updated["attrs"]:   # regex result takes priority
                op, threshold = val[0], int(val[1])
                updated["attrs"][col] = (op, threshold)
                updated["parsed_tokens"].append(
                    f'attr {col}{op}{threshold}  [from: LLM extraction]'
                )
                merged += 1
        if merged:
            print(f"  [LLM extractor] merged {merged} additional constraint(s).")
        return updated

    except Exception as e:
        print(f"  [LLM extractor] skipped: {e}")
        return parsed


# ---------------------------------------------------------------------------
# LLM Query Rewriter
# ---------------------------------------------------------------------------
# Instead of asking the LLM to output JSON (fragile), we ask it to rewrite
# the user's informal query into a structured English form that our regex
# parser already handles reliably.
#
# Architecture:
#   informal query  →  [LLM rewriter]  →  structured query  →  [parse_query]
#
# Falls back to parse_query on the original if LLM is unavailable.
# ---------------------------------------------------------------------------

_REWRITE_SYSTEM = """You are a query rewriter for a football scouting system.
Your job is to rewrite the user's informal scouting query into a clean structured form.

RULES:
1. Keep all position names (striker, midfielder, centre-back, winger, goalkeeper, etc.)
2. Convert descriptive adjectives into explicit attribute constraints using these FM24 attributes:
   pace, stamina, strength, work rate, aggression, tackling, passing, vision,
   dribbling, finishing, heading, off the ball, positioning, determination,
   composure, anticipation, decisions, technique, flair, balance, bravery,
   natural fitness, first touch, crossing, long shots, agility, aerial, reflexes
3. Use threshold format: "attribute >= value" (use 14 for good/high, 15 for excellent/world-class)
4. Keep age constraints as-is ("under 23", "older than 28", "teenager", "veteran")
5. Keep budget/value constraints ("valued under 10M EUR", "budget around 5M")
6. Keep wage constraints ("wages under 50K")
7. Keep nationality and foot preference as-is
8. Output ONLY the rewritten query, nothing else. No explanations.

EXAMPLES:
User: "high pressing tireless box-to-box midfielder"
Rewritten: "box-to-box midfielder, work rate >= 14, stamina >= 15, aggression >= 13"

User: "clinical poacher, cheap, teenager"
Rewritten: "striker, finishing >= 15, off the ball >= 14, under 19, valued under 5M EUR"

User: "solid no-nonsense centre back, veteran, affordable"
Rewritten: "centre-back, tackling >= 14, strength >= 14, heading >= 13, older than 30, valued under 5M EUR"

User: "creative playmaker with vision and flair, Brazilian"
Rewritten: "attacking midfielder, vision >= 15, flair >= 14, passing >= 14, Brazilian"

User: "pacy winger who can beat his man, under 22"
Rewritten: "winger, pace >= 14, dribbling >= 14, under 22"
"""

def rewrite_query(query):
    """
    Use the loaded local LLM to rewrite an informal query into a structured
    form that parse_query() can reliably parse.

    Returns (rewritten_query_str, was_rewritten_bool).
    If LLM is unavailable, returns (original_query, False).
    """
    try:
        if '_local_model' not in globals() or _local_model is None:
            return query, False

        messages = [
            {"role": "system",  "content": _REWRITE_SYSTEM},
            {"role": "user",    "content": f"User: \"{query}\"\nRewritten:"},
        ]
        text = _local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs  = _local_tokenizer([text], return_tensors="pt").to(_local_model.device)
        out_ids = _local_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.1,
            do_sample=False,
            pad_token_id=_local_tokenizer.eos_token_id,
        )
        new_ids    = out_ids[:, inputs["input_ids"].shape[-1]:]
        rewritten  = _local_tokenizer.decode(new_ids[0], skip_special_tokens=True).strip()

        # Clean up: strip any leading "Rewritten:" the model may echo
        rewritten = re.sub(r'^(?:rewritten|output|query)\s*:\s*', '', rewritten,
                           flags=re.IGNORECASE).strip('"\'\n ')

        return rewritten, True

    except Exception as e:
        print(f"  [rewrite_query] fallback to original: {e}")
        return query, False


def smart_parse(query, verbose=True):
    """
    Full pipeline: rewrite informal query with LLM (if available),
    then parse the structured result with parse_query().

    Returns the parsed dict (same format as parse_query).
    The parsed dict includes 'rewritten_query' key showing what the LLM produced.
    """
    rewritten, was_rewritten = rewrite_query(query)

    if was_rewritten and verbose:
        print(f"  [LLM rewrite] '{query}'")
        print(f"             → '{rewritten}'")

    parsed = parse_query(rewritten if was_rewritten else query)
    parsed['rewritten_query'] = rewritten if was_rewritten else None
    parsed['raw_query']       = query   # always keep original
    return parsed


---
## Section 7 -- Structured Filtering

Apply parsed constraints via Pandas filtering. Every applied and skipped constraint
is logged explicitly.

In [ ]:
def structured_filter(df, parsed):
    """
    Apply structured constraints from a parsed query dict.

    Returns: (filtered_df, applied_list, skipped_list)
    """
    mask = pd.Series(True, index=df.index)
    applied = []
    skipped = []

    # Age
    if parsed["age_min"] is not None:
        mask &= df["Age"] >= parsed["age_min"]
        applied.append(f"Age >= {parsed['age_min']}")
    if parsed["age_max"] is not None:
        mask &= df["Age"] <= parsed["age_max"]
        applied.append(f"Age <= {parsed['age_max']}")

    # Transfer Value
    if parsed["value_min"] is not None:
        mask &= df["Value_EUR"].notna() & (df["Value_EUR"] >= parsed["value_min"])
        applied.append(f"Value_EUR >= EUR{parsed['value_min']:,.0f}")
    if parsed["value_max"] is not None:
        # Exclude "Not for Sale" when a max value filter is applied
        mask &= df["Value_EUR"].notna() & ~df["Not_For_Sale"] & (df["Value_EUR"] <= parsed["value_max"])
        applied.append(f"Value_EUR <= EUR{parsed['value_max']:,.0f} (excl. Not-For-Sale)")

    # Wage
    if parsed["wage_min"] is not None:
        mask &= df["Wage_EUR_pm"].notna() & (df["Wage_EUR_pm"] >= parsed["wage_min"])
        applied.append(f"Wage_EUR_pm >= EUR{parsed['wage_min']:,.0f}/m")
    if parsed["wage_max"] is not None:
        mask &= df["Wage_EUR_pm"].notna() & (df["Wage_EUR_pm"] <= parsed["wage_max"])
        applied.append(f"Wage_EUR_pm <= EUR{parsed['wage_max']:,.0f}/m")

    # Foot
    if parsed["foot"]:
        mask &= df["Foot"] == parsed["foot"]
        applied.append(f"Foot == '{parsed['foot']}'")

    # Position
    if parsed["position_kws"]:
        pos_mask = df["Position"].apply(
            lambda p: any(kw.lower() in str(p).lower() for kw in parsed["position_kws"])
        )
        mask &= pos_mask
        applied.append(f"Position contains one of {parsed['position_kws']}")

    # Nationality
    if parsed["nationality"]:
        mask &= df["Nationality"].str.lower() == parsed["nationality"].lower()
        applied.append(f"Nationality == '{parsed['nationality']}'")

    # Style
    if parsed["style"]:
        mask &= df["Style"].str.lower() == parsed["style"].lower()
        applied.append(f"Style == '{parsed['style']}'")

    # Attribute thresholds
    for col, (op, threshold) in parsed["attrs"].items():
        if col not in df.columns:
            skipped.append(f"Attribute '{col}' not in dataset -- skipped")
            continue
        if op == ">":   mask &= df[col] > threshold
        elif op == ">=": mask &= df[col] >= threshold
        elif op == "<":  mask &= df[col] < threshold
        elif op == "<=": mask &= df[col] <= threshold
        elif op == "=":  mask &= df[col] == threshold
        applied.append(f"{col} {op} {threshold}")

    filtered = df[mask].copy()
    applied.append(f"-> {len(filtered):,} players passed (from {len(df):,})")
    return filtered, applied, skipped


# -- Test -------------------------------------------------------------------
# Parse a sample query so "parsed" is defined for the test below
parsed = parse_query(
    "Find a left-footed centre-back under 23 with passing > 15 "
    "and tackling >= 14, valued under EUR 10M"
)
filtered_df, applied_log, skipped_log = structured_filter(df, parsed)
print("Applied constraints:")
for a in applied_log:
    print(f"  [OK] {a}")
if skipped_log:
    print("Skipped constraints:")
    for s in skipped_log:
        print(f"  [WARN] {s}")

---
## Section 8 -- Semantic Retrieval

Player profiles are embedded using `all-MiniLM-L6-v2`. Query embeddings are compared
against profile embeddings using cosine similarity. FAISS is used if available;
otherwise sklearn provides an equivalent result.

> **Colab tip:** Embedding 42,541 profiles takes ~3-5 minutes on CPU. Embeddings are
> cached to disk so subsequent runs are instant.

In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading {EMBED_MODEL_NAME} on {DEVICE} ...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print("Model loaded")


In [ ]:
CACHE_PATH = os.path.join(DRIVE_ROOT, 'player_embeddings.pkl')
FAST_TEST  = False   # set True to embed only 10k players for quick testing

if os.path.exists(CACHE_PATH) and not FAST_TEST:
    print(f"Loading cached embeddings from {CACHE_PATH} ...")
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    player_embeddings = cache["embeddings"]
    cached_ids        = cache["player_ids"]
    if list(cached_ids) == list(df["player_id"]):
        print(f"Loaded {len(player_embeddings):,} cached embeddings")
    else:
        print("Cache mismatch -- re-embedding...")
        os.remove(CACHE_PATH)
else:
    embed_df = df.head(10_000) if FAST_TEST else df
    n = len(embed_df)
    print(f"Embedding {n:,} profiles on {DEVICE} ...")
    player_embeddings = embed_model.encode(
        embed_df["embed_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    if not FAST_TEST:
        cache = {"embeddings": player_embeddings, "player_ids": embed_df["player_id"]}
        with open(CACHE_PATH, "wb") as f:
            pickle.dump(cache, f)
        print(f"Saved to {CACHE_PATH}")
    else:
        print(f"[FAST_TEST] Embedded {n:,} players (not cached)")

print(f"Embedding matrix: {player_embeddings.shape}")


In [ ]:
# -- Build FAISS index ---------------------------------------------------
if FAISS_AVAILABLE:
    dim = player_embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dim)
    faiss_index.add(player_embeddings.astype("float32"))
    print(f"FAISS index built: {faiss_index.ntotal:,} vectors, dim={dim}")
else:
    faiss_index = None
    print("sklearn fallback active")

# Extract bare tokenizer + transformer
_st_module   = embed_model._first_module()
_tokenizer   = _st_module.tokenizer
_transformer = _st_module.auto_model.eval()
print(f"Transformer on: {next(_transformer.parameters()).device}")


def _mean_pool(model_output, attention_mask):
    token_emb = model_output.last_hidden_state
    mask_exp  = attention_mask.unsqueeze(-1).float().expand(token_emb.size())
    return torch.sum(token_emb * mask_exp, 1) / torch.clamp(mask_exp.sum(1), min=1e-9)


def embed_query(text):
    enc = _tokenizer(text, padding=True, truncation=True,
                     max_length=128, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        out = _transformer(**enc)
    pooled = _mean_pool(out, enc["attention_mask"])
    normed = F.normalize(pooled, p=2, dim=1)
    return normed.squeeze().cpu().numpy().astype("float32")


def semantic_search(query_text, candidate_df, candidate_embeddings, top_k=20):
    qvec = embed_query(query_text).reshape(1, -1)
    if FAISS_AVAILABLE and candidate_embeddings is player_embeddings:
        scores, indices = faiss_index.search(qvec, top_k)
        scores, indices = scores[0], indices[0]
        valid  = indices >= 0
        result = df.iloc[indices[valid]].copy()
        result["sem_score"] = scores[valid]
    else:
        sims    = cosine_similarity(qvec, candidate_embeddings)[0]
        top_idx = np.argsort(-sims)[:top_k]
        result  = candidate_df.iloc[top_idx].copy()
        result["sem_score"] = sims[top_idx]
    return result.sort_values("sem_score", ascending=False)


import time
t0 = time.time()
_v = embed_query("creative playmaker")
print(f"embed_query OK  shape={_v.shape}  ({time.time()-t0:.3f}s)")
t0 = time.time()
test_results = semantic_search(
    "creative left-footed playmaker with excellent vision and passing",
    df, player_embeddings, top_k=5,
)
print(f"semantic_search OK  ({time.time()-t0:.3f}s)")
print(test_results[["Player_Name", "Position", "Nationality", "Pas", "Vis", "sem_score"]].to_string(index=False))


In [ ]:
# Evidence columns shown in retrieval results
EVIDENCE_COLS = [
    "Player_Name", "Nationality", "Age", "Club_Name", "League",
    "Position", "Foot", "Style", "Value_EUR", "Wage_EUR_pm",
    "Acc", "Pac", "Pas", "Tec", "Sta", "Str", "Tck", "Dri", "Fin",
    "Vis", "Dec", "Ant", "OtB", "Mar", "Cro", "Hea", "Lon", "Bal",
    "sem_score", "player_id",
]


def hybrid_retrieve(query, top_k=10, verbose=True):
    """
    Full hybrid retrieval pipeline.

    Returns: (results_df, parsed, applied, skipped)
    """
    # Step 1: Parse (LLM rewrite -> regex parse if model loaded, else regex only)
    parsed = smart_parse(query, verbose=True)

    # Step 2: Structured filter
    filtered_df, applied, skipped = structured_filter(df, parsed)

    fallback_used = False
    if len(filtered_df) == 0:
        if verbose:
            print("[WARN] Structured filter returned 0 results. Falling back to full-dataset semantic search.")
        filtered_df = df
        applied.append("[WARN] Fallback: structured filter returned 0 -- semantic search on full dataset")
        fallback_used = True

    # Step 3: Get embeddings for candidate pool
    if len(filtered_df) == len(df) and not fallback_used:
        cand_embeddings = player_embeddings  # full matrix -- FAISS path available
    else:
        cand_idx = filtered_df.index.tolist()
        id_to_pos = {pid: pos for pos, pid in enumerate(df["player_id"].tolist())}
        positions = [id_to_pos[i] for i in cand_idx if i in id_to_pos]
        cand_embeddings = player_embeddings[positions]

    # Step 4: Semantic search over candidate pool
    results_df = semantic_search(
        parsed["semantic_query"],
        filtered_df.reset_index(drop=True),
        cand_embeddings,
        top_k=top_k,
    )

    # Step 5: Select evidence columns
    avail_cols = [c for c in EVIDENCE_COLS if c in results_df.columns]
    results_df = results_df[avail_cols].head(top_k)

    if verbose:
        print(f"\nQuery: {query}")
        print(f"Semantic residual: '{parsed['semantic_query']}'")
        n_constraints = len(applied) - 1  # last entry is the count line
        print(f"Constraints applied: {n_constraints}")
        print(f"Candidates after structured filter: {len(filtered_df):,}")
        print(f"\nTop-{top_k} hybrid results:")
        disp = [c for c in ["Player_Name","Age","Position","Nationality","Foot",
                             "Pas","Tec","Tck","Acc","Pac","sem_score"]
                if c in results_df.columns]
        print(results_df[disp].to_string(index=False))

    return results_df, parsed, applied, skipped


# -- Test -------------------------------------------------------------------
results, parsed_out, applied_log2, skipped_log2 = hybrid_retrieve(
    "Find a left-footed centre-back under 23 with passing > 15 "
    "and tackling >= 14, valued under EUR10M",
    top_k=10
)

---
## Section 10 -- Grounded Scouting Report Generation

Reports are generated from retrieved rows using a **template-based grounding approach**
-- the safe default that guarantees every cited value comes directly from the structured
data. An optional LLM generation path (via Groq free API) is provided but clearly marked
as secondary and optional.

**Design principle:** *no value in the report is invented.*

In [ ]:
def format_value(v_eur):
    """Format EUR value as a human-readable string."""
    if pd.isna(v_eur): return "unknown"
    if v_eur == 0:     return "free / EUR0"
    if v_eur >= 1_000_000: return f"EUR{v_eur/1_000_000:.1f}M"
    if v_eur >= 1_000:     return f"EUR{v_eur/1_000:.0f}K"
    return f"EUR{v_eur:.0f}"


def generate_grounded_report(query, results_df, applied, skipped, top_n=5):
    """
    Template-based grounded scouting report.
    Every numeric and categorical value cited comes directly from results_df.
    """
    lines = []
    lines.append("=" * 70)
    lines.append("SCOUTRAG SCOUTING REPORT")
    lines.append("=" * 70)
    lines.append(f"Query: {query}")
    lines.append("")
    lines.append("Constraints applied:")
    for a in applied:
        lines.append(f"  - {a}")
    if skipped:
        lines.append("Constraints not applied (reason):")
        for s in skipped:
            lines.append(f"  [WARN] {s}")
    lines.append("")
    lines.append(f"Top {min(top_n, len(results_df))} Recommended Players:")
    lines.append("-" * 70)

    for rank, (_, row) in enumerate(results_df.head(top_n).iterrows(), 1):
        name    = row.get("Player_Name", "Unknown")
        age     = row.get("Age", "?")
        pos     = row.get("Position", "?")
        nat     = row.get("Nationality", "?")
        club    = row.get("Club_Name", "?")
        league  = row.get("League", "?")
        foot    = row.get("Foot", "?")
        style   = row.get("Style", "?") if "Style" in row.index else "?"
        val     = format_value(row.get("Value_EUR", np.nan))
        wage_pm = row.get("Wage_EUR_pm", np.nan)
        wage_s  = f"EUR{wage_pm:,.0f}/month" if pd.notna(wage_pm) else "unknown"
        score   = row.get("sem_score", 0.0)

        # Top-6 outfield attributes present in the row
        attr_vals = {col: row[col] for col in OUTFIELD_ATTR_COLS
                     if col in row.index and pd.notna(row[col])}
        top_attrs = sorted(attr_vals.items(), key=lambda x: -x[1])[:6]
        attr_str  = ", ".join(f"{ATTR_READABLE.get(k,k)}={int(v)}" for k, v in top_attrs)

        lines.append(f"#{rank}  {name}  (Similarity: {score:.3f})")
        lines.append(f"    Age: {age} | Position: {pos} | Nationality: {nat}")
        lines.append(f"    Club: {club} ({league})")
        lines.append(f"    Foot: {foot} | Style: {style}")
        lines.append(f"    Value: {val} | Wage: {wage_s}")
        lines.append(f"    Key attributes: {attr_str}")
        lines.append("")

    lines.append("-" * 70)
    lines.append("Data source: Football Manager 2024 (game-derived synthetic data).")
    lines.append("Attribute scale: 1-20 (FM24 designer-set ratings).")
    lines.append("This report is grounded entirely in retrieved player rows.")
    lines.append("=" * 70)

    return "\n".join(lines)


# -- Generate a test report --------------------------------------------------
report = generate_grounded_report(
    "Find a left-footed centre-back under 23 with passing > 15 "
    "and tackling >= 14, valued under EUR10M",
    results, applied_log2, skipped_log2, top_n=5
)
print(report)

In [ ]:
# ---------------------------------------------------------------------------
# Optional: LLM-based report generation
# ---------------------------------------------------------------------------
# Choose a backend. The template report above is always the primary output;
# this is an optional enhancement for demo / qualitative evaluation.
#
# Backends:
#   "groq"    -- Groq Cloud API (free tier, ~instant, needs GROQ_API_KEY)
#   "hf_api"  -- HuggingFace Inference API (free tier, no GPU needed)
#   "local"   -- Load model directly on Colab T4/A100 via transformers
#                (no API key; Qwen2.5-7B-Instruct in 4-bit fits on T4)
#   "none"    -- Disable LLM generation entirely
# ---------------------------------------------------------------------------
LLM_BACKEND = "local"  # runs Qwen2.5-7B-Instruct on Colab T4, no API key needed

# API keys / tokens (only needed for the respective backend)
GROQ_API_KEY = ""       # get free key at console.groq.com
HF_TOKEN     = ""       # optional for hf_api; helps with rate limits

# Model names per backend
LLM_MODELS = {
    "groq":   "llama-3.1-8b-instant",      # Llama 3.1 8B -- fast & free
    "hf_api": "Qwen/Qwen2.5-7B-Instruct",  # Qwen 2.5 7B via HF Inference API
    "local":  "Qwen/Qwen2.5-3B-Instruct",   # 3B fits on T4 in fp16, no quantisation needed
}

USE_4BIT = False  # 3B model fits on T4 in fp16 -- no bitsandbytes needed


# ---------------------------------------------------------------------------
# Backend implementations
# ---------------------------------------------------------------------------

def _call_groq(system_prompt, user_prompt):
    try:
        from groq import Groq
    except ImportError:
        return "[groq not installed -- run: pip install groq]"
    client = Groq(api_key=GROQ_API_KEY)
    resp = client.chat.completions.create(
        model=LLM_MODELS["groq"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=600,
        temperature=0.2,
    )
    return resp.choices[0].message.content


def _call_hf_api(system_prompt, user_prompt):
    """HuggingFace Inference API (serverless, free tier)."""
    import requests
    model = LLM_MODELS["hf_api"]
    url   = f"https://api-inference.huggingface.co/models/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json"}
    if HF_TOKEN:
        headers["Authorization"] = f"Bearer {HF_TOKEN}"
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "max_tokens": 600,
        "temperature": 0.2,
    }
    r = requests.post(url, headers=headers, json=payload, timeout=60)
    if r.status_code != 200:
        return f"[HF API error {r.status_code}: {r.text[:200]}]"
    return r.json()["choices"][0]["message"]["content"]


# Module-level cache so we only load the local model once per session
_local_model    = None
_local_tokenizer = None

def _call_local(system_prompt, user_prompt):
    """Load Qwen2.5-3B-Instruct on the Colab GPU in fp16 -- no bitsandbytes needed."""
    global _local_model, _local_tokenizer
    if _local_model is None:
        print("Loading local model (first call -- ~2 min to download)...")
        from transformers import AutoTokenizer, AutoModelForCausalLM
        model_id = LLM_MODELS["local"]
        _local_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        _local_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        print("Local model loaded.")

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = _local_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs  = _local_tokenizer([text], return_tensors="pt").to(_local_model.device)
    out_ids = _local_model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.2,
        do_sample=True,
        pad_token_id=_local_tokenizer.eos_token_id,
    )
    # Strip the prompt tokens from the output
    new_ids = out_ids[:, inputs["input_ids"].shape[-1]:]
    return _local_tokenizer.decode(new_ids[0], skip_special_tokens=True)


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def generate_llm_report(query, results_df, top_n=5):
    """
    Generate a grounded LLM scouting report for the given query and
    retrieved player rows.  Returns a plain string.

    The LLM is constrained via the system prompt to only cite values that
    appear in the retrieved rows -- this mirrors the hallucination-grounding
    constraint we verify in the post-generation step.

    Data source note: all player attributes are from Football Manager 2024
    (synthetic game data), not real-world scouting data.
    """
    if LLM_BACKEND == "none":
        return "[LLM generation disabled -- set LLM_BACKEND to enable]"

    # Build a compact JSON context from the top-N retrieved rows
    cols  = ["Player_Name", "Age", "Position", "Nationality", "Club_Name",
             "Foot", "Style", "Value_EUR", "Wage_EUR_pm",
             "Acc", "Pac", "Pas", "Tec", "Sta", "Str", "Tck", "Dri", "Fin"]
    avail = [c for c in cols if c in results_df.columns]
    rows  = results_df.head(top_n)[avail].to_dict(orient="records")
    rows_str = json.dumps(rows, indent=2, default=str)

    system_prompt = (
        "You are a football scouting analyst assistant.\n"
        "You receive a scouting query and a JSON list of retrieved player records "
        "from a Football Manager 2024 dataset (synthetic game data, not real scouting data).\n"
        "Write a concise, grounded scouting report.\n\n"
        "STRICT RULES:\n"
        "1. Only cite player names, ages, clubs, and numeric attributes that appear "
        "verbatim in the provided JSON.\n"
        "2. Do NOT invent, estimate, or hallucinate any values.\n"
        "3. If you mention a numeric stat, copy it exactly from the JSON.\n"
        "4. Keep the report under 400 words.\n"
        "5. Start with 1-2 sentences summarising the pool, then briefly profile the "
        "top 3 candidates.\n"
        "6. End with one sentence noting this is FM24 game data, not real scouting data.\n"
    )
    user_prompt = (
        f"Scouting query: {query}\n\n"
        f"Retrieved players (JSON):\n{rows_str}"
    )

    backend_fn = {
        "groq":   _call_groq,
        "hf_api": _call_hf_api,
        "local":  _call_local,
    }.get(LLM_BACKEND)

    if backend_fn is None:
        return f"[Unknown LLM_BACKEND '{LLM_BACKEND}']"

    try:
        return backend_fn(system_prompt, user_prompt)
    except Exception as e:
        return f"[LLM generation failed ({LLM_BACKEND}): {e}]"


# ---------------------------------------------------------------------------
# Demo call (uses whatever backend is set above)
# ---------------------------------------------------------------------------
_demo_query = (
    "Find a left-footed centre-back under 23 with passing > 15 "
    "and tackling >= 14, valued under EUR 10M"
)
llm_report = generate_llm_report(_demo_query, results, top_n=5)
print("--- LLM Report ---")
print(llm_report)


---
## Section 11 -- Post-Generation Grounding Verification

**This is the core technical contribution of the project.**

After generating a scouting report, we automatically verify that:
1. Every player name mentioned in the report is among the retrieved rows.
2. Every numeric attribute value cited (e.g. "passing=17") matches the source row exactly.
3. We compute a **grounding score**: fraction of verifiable claims that are supported.

This gives an operational hallucination rate even without a human annotator.

In [ ]:
def verify_grounding(report_text, results_df):
    """
    Verify grounding of a generated report against retrieved player rows.

    Checks:
    - Player name mentions -> must appear in results_df["Player_Name"]
    - Numeric attribute citations -> "attribute=VALUE" must match the actual row value

    Returns dict with:
      supported_facts, mismatched_facts, unsupported_players,
      total_checked, grounding_score, hallucination_count
    """
    vr = {
        "supported_facts":     [],
        "mismatched_facts":    [],
        "unsupported_players": [],
        "total_checked":       0,
        "grounding_score":     1.0,
        "hallucination_count": 0,
    }

    # Build player lookup: lowercase name -> row
    player_lookup = {}
    for _, row in results_df.iterrows():
        pname = str(row.get("Player_Name", "")).lower()
        player_lookup[pname] = row
    retrieved_names_lower = set(player_lookup.keys())

    # Check 1: Player name mentions
    for pname_lower, row in player_lookup.items():
        pname_display = row.get("Player_Name", pname_lower)
        if pname_lower in report_text.lower():
            vr["supported_facts"].append(f"[OK] Player mentioned: {pname_display}")
            vr["total_checked"] += 1

    # Flag capitalised word sequences (2-4 words) not matching any retrieved player
    candidate_names = re.findall(
        r"(?:[A-Z][a-z]+\s+){1,3}[A-Z][a-z]+", report_text
    )
    for cname in candidate_names:
        cname_lower = cname.lower().strip()
        found = any(cname_lower in rn or rn in cname_lower for rn in retrieved_names_lower)
        if not found and len(cname.split()) >= 2:
            vr["unsupported_players"].append(cname)
            vr["hallucination_count"] += 1
            vr["total_checked"] += 1

    # Check 2: Numeric attribute citations (pattern: "attribute=VALUE" or "attribute VALUE")
    attr_patterns = [(readable, abbr) for abbr, readable in ATTR_READABLE.items()]
    attr_patterns.sort(key=lambda x: -len(x[0]))
    readable_to_col = {r.lower(): a for r, a in attr_patterns}

    numeric_claims = re.findall(
        r"(" + "|".join(re.escape(r) for r, _ in attr_patterns) + r")[=:]\s*(\d+)",
        report_text.lower()
    )

    for attr_readable_str, value_str in numeric_claims:
        col = readable_to_col.get(attr_readable_str.lower())
        if col is None or col not in results_df.columns:
            vr["total_checked"] += 1
            vr["unsupported_players"].append(f"Unknown attribute: {attr_readable_str}")
            vr["hallucination_count"] += 1
            continue

        cited_val = int(value_str)
        matched_rows = results_df[results_df[col] == cited_val]
        vr["total_checked"] += 1
        if len(matched_rows) > 0:
            vr["supported_facts"].append(
                f"[OK] {attr_readable_str}={cited_val} matches {len(matched_rows)} retrieved player(s)"
            )
        else:
            actual_vals = results_df[col].tolist()
            vr["mismatched_facts"].append(
                f"[FAIL] {attr_readable_str}={cited_val} not in retrieved rows "
                f"(actual: {actual_vals[:5]})"
            )
            vr["hallucination_count"] += 1

    # Grounding score
    if vr["total_checked"] > 0:
        vr["grounding_score"] = 1.0 - (vr["hallucination_count"] / vr["total_checked"])

    return vr


def print_grounding_report(vr):
    """Print a formatted grounding verification report."""
    print("=" * 60)
    print("GROUNDING VERIFICATION REPORT")
    print("=" * 60)
    print(f"Total claims checked   : {vr['total_checked']}")
    print(f"Supported facts        : {len(vr['supported_facts'])}")
    print(f"Mismatched facts       : {len(vr['mismatched_facts'])}")
    print(f"Unsupported mentions   : {len(vr['unsupported_players'])}")
    print(f"Hallucination count    : {vr['hallucination_count']}")
    print(f"Grounding score        : {vr['grounding_score']:.2%}")
    if vr["supported_facts"]:
        print("\nSupported facts (sample):")
        for f in vr["supported_facts"][:5]:
            print(f"  {f}")
    if vr["mismatched_facts"]:
        print("\nMismatched facts:")
        for f in vr["mismatched_facts"]:
            print(f"  {f}")
    if vr["unsupported_players"]:
        print("\nUnsupported mentions:")
        for f in vr["unsupported_players"][:5]:
            print(f"  [FAIL] {f}")
    print("=" * 60)


# -- Test on template report (should have near-perfect grounding) ------------
vr_template = verify_grounding(report, results)
print_grounding_report(vr_template)

---
## Section 12 -- Evaluation Query Set

40 scouting queries covering all constraint types:
numerical, categorical, financial, semantic/tactical, multi-constraint, difficult/edge-case.

In [ ]:
EVAL_QUERIES = [
    # Type A: Pure numerical constraints
    {"id": 1,  "type": "numerical",   "notes": "Simple age + position",
     "query": "Find strikers under 21 years old",
     "expected_pos": ["ST (C)"], "constraints": {"age_max": 20}},
    {"id": 2,  "type": "numerical",   "notes": "GK age filter",
     "query": "Goalkeepers over 30 years old",
     "expected_pos": ["GK"], "constraints": {"age_min": 30}},
    {"id": 3,  "type": "numerical",   "notes": "High-threshold single attribute",
     "query": "Players with pace over 18",
     "expected_pos": None, "constraints": {"attrs": {"Pac": (">", 18)}}},
    {"id": 4,  "type": "numerical",   "notes": "Age range + position",
     "query": "Defenders aged between 22 and 27",
     "expected_pos": ["D (C)"], "constraints": {"age_min": 22, "age_max": 27}},
    {"id": 5,  "type": "numerical",   "notes": "Multi-attribute numerical",
     "query": "Players with stamina over 17 and work rate over 16",
     "expected_pos": None, "constraints": {"attrs": {"Sta": (">", 17), "Wor": (">", 16)}}},

    # Type B: Categorical constraints
    {"id": 6,  "type": "categorical", "notes": "Foot + position",
     "query": "Left-footed wingers",
     "expected_pos": ["AM (RL)"], "constraints": {"foot": "Left"}},
    {"id": 7,  "type": "categorical", "notes": "Nationality + position",
     "query": "Brazilian central midfielders",
     "expected_pos": ["M (C)"], "constraints": {"nationality": "Brazilian"}},
    {"id": 8,  "type": "categorical", "notes": "Style + position",
     "query": "Physical style strikers",
     "expected_pos": ["ST (C)"], "constraints": {"style": "Physical"}},
    {"id": 9,  "type": "categorical", "notes": "Nationality + foot",
     "query": "French left-footed players",
     "expected_pos": None, "constraints": {"nationality": "French", "foot": "Left"}},
    {"id": 10, "type": "categorical", "notes": "Style + position",
     "query": "Creative style attacking midfielders",
     "expected_pos": ["AM (C)"], "constraints": {"style": "Creative"}},

    # Type C: Financial constraints
    {"id": 11, "type": "financial",   "notes": "Budget defender search",
     "query": "Centre-backs valued under EUR5M",
     "expected_pos": ["D (C)"], "constraints": {"value_max": 5_000_000}},
    {"id": 12, "type": "financial",   "notes": "Youth bargain striker",
     "query": "Strikers valued under EUR1M under 20 years old",
     "expected_pos": ["ST (C)"], "constraints": {"value_max": 1_000_000, "age_max": 19}},
    {"id": 13, "type": "financial",   "notes": "Low-wage GK",
     "query": "Goalkeepers with wage under EUR5000 per month",
     "expected_pos": ["GK"], "constraints": {"wage_max": 5000}},
    {"id": 14, "type": "financial",   "notes": "Very low budget midfielder",
     "query": "Midfielders valued under EUR500K",
     "expected_pos": ["M (C)"], "constraints": {"value_max": 500_000}},
    {"id": 15, "type": "financial",   "notes": "Elite tier -- few results",
     "query": "Players valued over EUR100M",
     "expected_pos": None, "constraints": {"value_min": 100_000_000}},

    # Type D: Semantic / tactical constraints
    {"id": 16, "type": "semantic",    "notes": "Tactical profile -- semantic dominant",
     "query": "High-pressing midfielder with excellent stamina and work rate",
     "expected_pos": ["M (C)", "DM"], "constraints": {}},
    {"id": 17, "type": "semantic",    "notes": "Modern CB archetype",
     "query": "Ball-playing centre-back comfortable in possession",
     "expected_pos": ["D (C)"], "constraints": {}},
    {"id": 18, "type": "semantic",    "notes": "Semantic winger profile",
     "query": "Creative winger with flair and dribbling ability",
     "expected_pos": ["AM (RL)"], "constraints": {}},
    {"id": 19, "type": "semantic",    "notes": "Sweeper-keeper profile",
     "query": "Goalkeeper who is commanding and good with the ball",
     "expected_pos": ["GK"], "constraints": {}},
    {"id": 20, "type": "semantic",    "notes": "Target striker profile",
     "query": "Powerful target man striker who wins aerial duels",
     "expected_pos": ["ST (C)"], "constraints": {}},
    {"id": 21, "type": "semantic",    "notes": "Regista role",
     "query": "Deep-lying playmaker with vision and passing range",
     "expected_pos": ["DM", "M (C)"], "constraints": {}},
    {"id": 22, "type": "semantic",    "notes": "Box-to-box midfielder",
     "query": "Box-to-box midfielder with stamina aggression and goals",
     "expected_pos": ["M (C)"], "constraints": {}},
    {"id": 23, "type": "semantic",    "notes": "Attacking full-back",
     "query": "Pacey full-back who loves to overlap and cross",
     "expected_pos": ["D/WB (L)", "D/WB (R)"], "constraints": {}},

    # Type E: Multi-constraint queries
    {"id": 24, "type": "multi",       "notes": "Classic multi-constraint CB search",
     "query": "Left-footed centre-back under 23 with passing > 15 and tackling >= 14 valued under EUR10M",
     "expected_pos": ["D (C)"],
     "constraints": {"foot": "Left", "age_max": 22, "value_max": 10_000_000,
                     "attrs": {"Pas": (">", 15), "Tck": (">=", 14)}}},
    {"id": 25, "type": "multi",       "notes": "National team striker search",
     "query": "Brazilian right-footed striker under 25 with finishing over 15 and pace over 14",
     "expected_pos": ["ST (C)"],
     "constraints": {"nationality": "Brazilian", "age_max": 24,
                     "attrs": {"Fin": (">", 15), "Pac": (">", 14)}}},
    {"id": 26, "type": "multi",       "notes": "Experienced DM profile",
     "query": "German defensive midfielder over 28 with tackling > 14 and composure > 14",
     "expected_pos": ["DM"],
     "constraints": {"nationality": "German", "age_min": 28,
                     "attrs": {"Tck": (">", 14), "Cmp": (">", 14)}}},
    {"id": 27, "type": "multi",       "notes": "Classic CAM profile",
     "query": "Left-footed attacking midfielder aged 20 to 26 with technique over 15 and vision over 14",
     "expected_pos": ["AM (C)"],
     "constraints": {"foot": "Left", "age_min": 20, "age_max": 26,
                     "attrs": {"Tec": (">", 15), "Vis": (">", 14)}}},
    {"id": 28, "type": "multi",       "notes": "Youth target man",
     "query": "Physical style striker under 24 with strength > 15 and heading > 14",
     "expected_pos": ["ST (C)"],
     "constraints": {"style": "Physical", "age_max": 23,
                     "attrs": {"Str": (">", 15), "Hea": (">", 14)}}},
    {"id": 29, "type": "multi",       "notes": "Spanish La Masia style winger",
     "query": "Left-footed Spanish winger aged between 18 and 24 with dribbling over 15",
     "expected_pos": ["AM (RL)"],
     "constraints": {"nationality": "Spanish", "foot": "Left",
                     "age_min": 18, "age_max": 24,
                     "attrs": {"Dri": (">", 15)}}},
    {"id": 30, "type": "multi",       "notes": "Ligue 2 scouting scenario",
     "query": "French central midfielder under 22 with passing > 14 and stamina > 14 valued under EUR20M",
     "expected_pos": ["M (C)"],
     "constraints": {"nationality": "French", "age_max": 21, "value_max": 20_000_000,
                     "attrs": {"Pas": (">", 14), "Sta": (">", 14)}}},

    # Type F: Difficult / edge-case queries
    {"id": 31, "type": "difficult",   "notes": "No 'Scandinavian' nationality key -- parser fails",
     "query": "Find me a Scandinavian goalkeeper under 25",
     "expected_pos": ["GK"], "constraints": {}},
    {"id": 32, "type": "difficult",   "notes": "Vague financial + multi-role + semantic",
     "query": "Cheap young creative playmaker who can play anywhere in midfield",
     "expected_pos": None, "constraints": {}},
    {"id": 33, "type": "difficult",   "notes": "Pure semantic -- tests embedding quality",
     "query": "A leader who can anchor the defence and organise the backline",
     "expected_pos": ["D (C)"], "constraints": {}},
    {"id": 34, "type": "difficult",   "notes": "High threshold -- very few results",
     "query": "Players with aggression over 18",
     "expected_pos": None, "constraints": {"attrs": {"Agg": (">", 18)}}},
    {"id": 35, "type": "difficult",   "notes": "GK-specific attributes at high thresholds",
     "query": "Goalkeeper under 22 with reflexes > 16 and handling > 15",
     "expected_pos": ["GK"],
     "constraints": {"age_max": 21, "attrs": {"Ref": (">", 16), "Han": (">", 15)}}},
    {"id": 36, "type": "difficult",   "notes": "Multi-position player -- FM position strings complex",
     "query": "Left-back who can fill in as a winger and has crossing > 15",
     "expected_pos": ["D/WB (L)"],
     "constraints": {"attrs": {"Cro": (">", 15)}}},
    {"id": 37, "type": "difficult",   "notes": "Very restrictive -- tests graceful degradation",
     "query": "Cheap striker valued under EUR100K who is under 19",
     "expected_pos": ["ST (C)"],
     "constraints": {"age_max": 18, "value_max": 100_000}},
    {"id": 38, "type": "difficult",   "notes": "Style + semantic combination",
     "query": "An intelligent defensive midfielder who reads the game well",
     "expected_pos": ["DM"],
     "constraints": {"style": "Intelligent"}},
    {"id": 39, "type": "difficult",   "notes": "Free agent constraint via status",
     "query": "Free agent striker aged between 25 and 32 with finishing > 14",
     "expected_pos": ["ST (C)"],
     "constraints": {"age_min": 25, "age_max": 32, "attrs": {"Fin": (">", 14)}}},
    {"id": 40, "type": "difficult",   "notes": "OR nationality -- parser handles one only",
     "query": "Brazilian or Argentine creative midfielder under 21",
     "expected_pos": ["M (C)", "AM (C)"], "constraints": {}},
]

print(f"Total evaluation queries: {len(EVAL_QUERIES)}")
from collections import Counter
print("\nQuery type distribution:")
for t, n in Counter(q["type"] for q in EVAL_QUERIES).items():
    print(f"  {t:12s}: {n}")

---
## Section 13 -- Evaluation Metrics

We evaluate:
- **Constraint satisfaction rate (CSR):** fraction of top-k results satisfying all parsed constraints.
- **Result count:** whether the system returns enough candidates.
- **Grounding score:** from the post-generation verification step.
- **Method comparison:** structured-only vs vector-only vs hybrid.

In [ ]:
def check_constraint_satisfaction(row, parsed):
    """Return True if row satisfies all parsed constraints."""
    if parsed["age_min"] and row["Age"] < parsed["age_min"]:    return False
    if parsed["age_max"] and row["Age"] > parsed["age_max"]:    return False
    if parsed["value_max"] and pd.notna(row.get("Value_EUR")):
        if row["Value_EUR"] > parsed["value_max"]:               return False
    if parsed["value_min"] and pd.notna(row.get("Value_EUR")):
        if row["Value_EUR"] < parsed["value_min"]:               return False
    if parsed["wage_max"] and pd.notna(row.get("Wage_EUR_pm")):
        if row["Wage_EUR_pm"] > parsed["wage_max"]:              return False
    if parsed["foot"] and row["Foot"] != parsed["foot"]:         return False
    if parsed["nationality"]:
        if row["Nationality"].lower() != parsed["nationality"].lower(): return False
    if parsed["style"]:
        if str(row.get("Style","")).lower() != parsed["style"].lower(): return False
    for col, (op, threshold) in parsed["attrs"].items():
        if col not in row.index: continue
        v = row[col]
        if op == ">"  and not (v >  threshold): return False
        if op == ">=" and not (v >= threshold): return False
        if op == "<"  and not (v <  threshold): return False
        if op == "<=" and not (v <= threshold): return False
        if op == "="  and not (v == threshold): return False
    return True


def evaluate_query(q_entry, top_k=10):
    """Run a single evaluation query and return metrics dict."""
    query = q_entry["query"]

    # Hybrid
    results_h, parsed_h, applied_h, skipped_h = hybrid_retrieve(query, top_k=top_k, verbose=False)

    # Structured-only baseline
    filtered_s, _, _ = structured_filter(df, parsed_h)
    results_s = filtered_s.head(top_k).copy() if len(filtered_s) > 0 else pd.DataFrame()
    if len(results_s) > 0:
        results_s["sem_score"] = 0.0

    # Vector-only baseline
    results_v = semantic_search(parsed_h["semantic_query"], df, player_embeddings, top_k=top_k)

    # CSR for each method
    def csr(res_df):
        if len(res_df) == 0: return 0.0
        sat = sum(check_constraint_satisfaction(row, parsed_h) for _, row in res_df.iterrows())
        return sat / len(res_df)

    csr_h = csr(results_h)
    csr_s = csr(results_s)
    csr_v = csr(results_v)

    # Grounding score (template report)
    rpt = generate_grounded_report(query, results_h, applied_h, skipped_h,
                                   top_n=min(5, len(results_h)))
    vr  = verify_grounding(rpt, results_h)

    return {
        "id": q_entry["id"], "query": query, "type": q_entry["type"],
        "n_results_hybrid":     len(results_h),
        "n_results_structured": len(results_s),
        "n_results_vector":     len(results_v),
        "csr_hybrid":     round(csr_h, 3),
        "csr_structured": round(csr_s, 3),
        "csr_vector":     round(csr_v, 3),
        "grounding_score":     round(vr["grounding_score"], 3),
        "hallucination_count": vr["hallucination_count"],
        "n_constraints_applied": len(applied_h) - 1,
        "n_constraints_skipped": len(skipped_h),
        "notes": q_entry.get("notes", ""),
    }


print("Evaluation function ready. Run the next cell to evaluate all 40 queries.")
print("(May take several minutes due to embedding calls for each query.)")

In [ ]:
# -- Run evaluation on all 40 queries ----------------------------------------
print("Running evaluation...")
eval_results = []
for q in EVAL_QUERIES:
    r = evaluate_query(q, top_k=10)
    eval_results.append(r)
    print(f"  [{r['id']:2d}] {r['type']:10s} | "
          f"CSR_h={r['csr_hybrid']:.2f} CSR_s={r['csr_structured']:.2f} "
          f"CSR_v={r['csr_vector']:.2f} | "
          f"grnd={r['grounding_score']:.2f} | n={r['n_results_hybrid']}")

eval_df = pd.DataFrame(eval_results)
print(f"\nEvaluation complete: {len(eval_df)} queries")

In [ ]:
# -- Aggregate results -------------------------------------------------------
print("\n=== AGGREGATE EVALUATION RESULTS ===")
print(f"{'Metric':<40} {'Structured':>12} {'Vector':>12} {'Hybrid':>12}")
print("-" * 76)
print(f"{'Mean CSR':<40} "
      f"{eval_df['csr_structured'].mean():>12.3f} "
      f"{eval_df['csr_vector'].mean():>12.3f} "
      f"{eval_df['csr_hybrid'].mean():>12.3f}")
print(f"{'Queries with >= 1 result':<40} "
      f"{(eval_df['n_results_structured']>0).sum():>12d} "
      f"{(eval_df['n_results_vector']>0).sum():>12d} "
      f"{(eval_df['n_results_hybrid']>0).sum():>12d}")
print(f"{'Mean result count':<40} "
      f"{eval_df['n_results_structured'].mean():>12.1f} "
      f"{eval_df['n_results_vector'].mean():>12.1f} "
      f"{eval_df['n_results_hybrid'].mean():>12.1f}")
print(f"{'Mean grounding score (hybrid)':<40} {'--':>12} {'--':>12} "
      f"{eval_df['grounding_score'].mean():>12.3f}")
print(f"{'Mean hallucination count (hybrid)':<40} {'--':>12} {'--':>12} "
      f"{eval_df['hallucination_count'].mean():>12.2f}")

print("\n=== BY QUERY TYPE ===")
print(eval_df.groupby("type")[
    ["csr_hybrid","csr_structured","csr_vector","grounding_score"]
].mean().round(3).to_string())

In [ ]:
# -- Visualise results -------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: CSR comparison by method
methods   = ["Structured", "Vector", "Hybrid"]
mean_csrs = [eval_df["csr_structured"].mean(),
             eval_df["csr_vector"].mean(),
             eval_df["csr_hybrid"].mean()]
colors    = ["#4C72B0", "#DD8452", "#55A868"]
axes[0].bar(methods, mean_csrs, color=colors, width=0.5)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Mean Constraint Satisfaction Rate")
axes[0].set_title("CSR: Structured vs Vector vs Hybrid")
for i, v in enumerate(mean_csrs):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

# Plot 2: CSR by query type (hybrid)
type_csr = eval_df.groupby("type")["csr_hybrid"].mean().sort_values()
type_csr.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_xlim(0, 1.05)
axes[1].set_xlabel("Mean CSR (Hybrid)")
axes[1].set_title("Hybrid CSR by Query Type")

# Plot 3: Grounding score distribution
axes[2].hist(eval_df["grounding_score"], bins=10, color="#C44E52",
             edgecolor="black", alpha=0.8)
axes[2].set_xlabel("Grounding Score")
axes[2].set_ylabel("Number of Queries")
axes[2].set_title("Grounding Score Distribution (Hybrid)")
mean_gs = eval_df["grounding_score"].mean()
axes[2].axvline(mean_gs, color="black", linestyle="--", label=f"Mean={mean_gs:.2f}")
axes[2].legend()

plt.suptitle("ScoutRAG Evaluation Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("scoutrag_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to scoutrag_evaluation.png")

---
## Section 14 -- Baselines and Ablation

Three retrieval configurations compared:
- **Structured-only:** pandas filters; no semantic component.
- **Vector-only:** pure embedding search; ignores structured constraints.
- **Hybrid:** structured filter then semantic search over filtered pool. Our proposed method.

In [ ]:
# -- Full comparison table ----------------------------------------------------
comparison_cols = ["id", "type", "csr_structured", "csr_vector", "csr_hybrid",
                   "n_results_structured", "n_results_vector", "n_results_hybrid",
                   "grounding_score"]
print(eval_df[comparison_cols].to_string(index=False))

In [ ]:
# -- Ablation: where does hybrid most improve over vector-only? ---------------
eval_df["hybrid_gain"] = eval_df["csr_hybrid"] - eval_df["csr_vector"]

print("Queries where Hybrid most outperforms Vector-only:")
best_gains = eval_df.nlargest(8, "hybrid_gain")[
    ["id", "query", "csr_vector", "csr_hybrid", "hybrid_gain"]
]
print(best_gains.to_string(index=False))

print("\nQueries where Vector-only matches or beats Hybrid:")
neutral = eval_df[eval_df["hybrid_gain"] <= 0][
    ["id", "query", "csr_vector", "csr_hybrid", "hybrid_gain"]
]
print(neutral.to_string(index=False))

In [ ]:
# -- Zero-result failure cases ------------------------------------------------
failures = eval_df[eval_df["n_results_hybrid"] == 0][["id", "type", "query", "notes"]]
print(f"Queries with 0 hybrid results: {len(failures)}")
if len(failures):
    print(failures.to_string(index=False))
else:
    print("No queries returned 0 results.")

---
## Section 15 -- Error Analysis

An honest account of where and why the system fails.

In [ ]:
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

print("\n1. CONSTRAINT PARSER LIMITATIONS")
print("  - OR conditions (e.g. 'Brazilian or Argentine') extract only the first nationality.")
print("  - Vague terms like 'cheap', 'young', 'experienced' map to no threshold.")
print("  - Regional descriptors ('Scandinavian', 'South American') are not in the")
print("    nationality list and are silently passed to semantic retrieval, which")
print("    handles them weakly because profiles encode nationality as a word, not")
print("    a regional grouping.")
print("  - Multi-position queries ('can play CB and DM') resolve to one position category.")

print("\n2. DATASET LIMITATIONS")
n_no_pos = (df["Position"] == "Unknown").sum()
print(f"  - {n_no_pos:,} players have no 'Position' entry -- they participate in")
print(f"    semantic search but cannot satisfy position constraints.")
print(f"  - {df['Not_For_Sale'].sum():,} 'Not for Sale' players are excluded when a")
print(f"    value_max filter is applied, which removes some star players.")
print(f"  - Transfer value ranges (e.g. 'EUR14K - EUR140K') -> we parse the upper bound.")
print(f"    This overstates value for cheap players, making value_max filters more")
print(f"    restrictive than expected.")
print(f"  - Wage is missing for {df['Wage_EUR_pm'].isna().sum():,} players "
      f"({100*df['Wage_EUR_pm'].isna().mean():.1f}%).")
print(f"    Wage-constrained queries silently exclude these players.")
print(f"  - FM24 ratings are game attributes. Correlation with real-world ability")
print(f"    is not validated.")

print("\n3. SEMANTIC RETRIEVAL LIMITATIONS")
print("  - 'all-MiniLM-L6-v2' is general-purpose; football-specific jargon")
print("    ('regista', 'mezzala', 'inverted winger') may not embed precisely.")
print("  - Profiles are built from attributes, not free text scouting notes.")
print("    Queries about 'leadership' or 'mentality' map onto Ldr/Det/Bra attributes")
print("    indirectly -- the semantic matching is approximate.")

print("\n4. GROUNDING VERIFIER LIMITATIONS")
print("  - The verifier uses regex. Complex LLM sentences expressing numbers in")
print("    words (e.g., 'a pace of around sixteen') will be missed.")
print("  - Template reports are grounded by construction -- the verifier is most")
print("    informative when applied to LLM-generated free-text reports.")
print("  - Short player names (e.g., 'Kane') may match common English words,")
print("    producing false-positive name matches.")

print("\n5. CSR BREAKDOWN BY QUERY TYPE")
print(eval_df.groupby("type")["csr_hybrid"].agg(
    ["mean", "min", "max", "count"]
).round(3).to_string())

---
## Section 16 -- Demo: `scout()` Function

A single clean entry point running the full pipeline.

In [ ]:
def scout(query, top_k=10, show_grounding=True):
    """
    ScoutRAG main demo function.

    Parameters
    ----------
    query          : natural-language scouting query
    top_k          : number of candidates to retrieve and report
    show_grounding : whether to print the grounding verification report

    Returns
    -------
    results_df, report, grounding
    """
    print("\n" + "=" * 70)
    print("  ScoutRAG -- Constrained Hybrid Retrieval Scouting Assistant")
    print("=" * 70)
    print(f"  Query: {query}")
    print("=" * 70 + "\n")

    # Hybrid retrieval
    results_df, parsed, applied, skipped = hybrid_retrieve(query, top_k=top_k, verbose=False)

    # Show parsed constraints
    print("Parsed Constraints:")
    if parsed["parsed_tokens"]:
        for tok in parsed["parsed_tokens"]:
            print(f"  [OK] {tok}")
    else:
        print("  (no structured constraints detected -- full semantic retrieval)")
    if skipped:
        print("Skipped Constraints:")
        for s in skipped:
            print(f"  [WARN] {s}")
    print(f"\nCandidates after structured filter: {len(results_df)} returned (top {top_k})\n")

    # Results table
    disp = ["Player_Name", "Age", "Position", "Nationality", "Foot", "Style",
            "Pac", "Pas", "Tec", "Fin", "Tck", "Str", "Sta", "Vis", "sem_score"]
    disp = [c for c in disp if c in results_df.columns]
    print(results_df[disp].to_string(index=False))

    # Grounded report
    report = generate_grounded_report(query, results_df, applied, skipped,
                                      top_n=min(5, len(results_df)))
    print("\n" + report)

    # Grounding verification
    grounding = verify_grounding(report, results_df)
    if show_grounding:
        print_grounding_report(grounding)

    return results_df, report, grounding


# -- Demo 1: multi-constraint -------------------------------------------------
_ = scout(
    "Left-footed centre-back under 23 with passing > 15 "
    "and tackling >= 14, valued under EUR10M",
    top_k=8
)

In [ ]:
# -- Demo 2: semantic / tactical query ----------------------------------------
_ = scout(
    "High-pressing box-to-box midfielder with excellent stamina and work rate",
    top_k=8
)

In [ ]:
# -- Demo 3: budget youth search ----------------------------------------------
_ = scout(
    "Young Brazilian winger under 20 with dribbling over 15 valued under EUR5M",
    top_k=8
)

---
## Section 17 -- Final Report-Ready Summary

Concise outputs for use in the final CS 455 project report.

In [ ]:
print("=" * 70)
print("SCOUTRAG -- FINAL SUMMARY FOR CS 455 REPORT")
print("=" * 70)

print("\n--- SYSTEM ARCHITECTURE ---")
arch = (
    "ScoutRAG is a constrained hybrid retrieval system for football player\n"
    "scouting over the FM24 game-derived tabular dataset.\n\n"
    "Pipeline:\n"
    "  1. Rule-based constraint parser   -> extracts age / value / wage /\n"
    "     position / foot / nationality / attribute thresholds from query.\n"
    "  2. Pandas structured filter       -> applies parsed constraints to\n"
    "     42,541 player rows, producing a constrained candidate pool.\n"
    "  3. Sentence-transformer embedding -> all-MiniLM-L6-v2 embeds both\n"
    "     player profiles and the semantic residual of the query.\n"
    "  4. Cosine similarity (sklearn /   -> retrieves top-k players from\n"
    "     FAISS)                            the candidate pool.\n"
    "  5. Template-based report generator-> grounded scouting report citing\n"
    "     only retrieved row values.\n"
    "  6. Post-generation verifier       -> checks cited player names and\n"
    "     numeric attributes against source rows; computes grounding score.\n"
)
print(arch)

print("--- DATASET SUMMARY ---")
print(f"  Source         : Football Manager 2024 (FM24) in-game export")
print(f"  Nature         : Game-derived synthetic data -- NOT real-world scouting data")
print(f"  Total players  : {len(df):,}")
print(f"  Columns        : {len(df.columns)}")
print(f"  Attribute scale: 1-20 integer (designer-set game ratings)")
print(f"  Key columns    : Player_Name, Nationality, Age, Club_Name, League,")
print(f"                   Position, Foot, Style, Value_EUR, Wage_EUR_pm,")
print(f"                   + {len(ALL_ATTR_COLS)} numeric attributes")

print("\n--- EVALUATION RESULTS TABLE ---")
summary = eval_df.groupby("type").agg(
    n_queries=("id", "count"),
    mean_csr_structured=("csr_structured", "mean"),
    mean_csr_vector=("csr_vector", "mean"),
    mean_csr_hybrid=("csr_hybrid", "mean"),
    mean_grounding=("grounding_score", "mean"),
).round(3)
print(summary.to_string())

print("\n--- BASELINE COMPARISON (all 40 queries) ---")
print(f"  Metric                   Structured  Vector-only   Hybrid")
print(f"  Mean CSR                 "
      f"{eval_df['csr_structured'].mean():.3f}        "
      f"{eval_df['csr_vector'].mean():.3f}         "
      f"{eval_df['csr_hybrid'].mean():.3f}")
print(f"  Queries with results     "
      f"{(eval_df['n_results_structured']>0).sum()}/40          "
      f"{(eval_df['n_results_vector']>0).sum()}/40           "
      f"{(eval_df['n_results_hybrid']>0).sum()}/40")
print(f"  Mean result count        "
      f"{eval_df['n_results_structured'].mean():.1f}          "
      f"{eval_df['n_results_vector'].mean():.1f}           "
      f"{eval_df['n_results_hybrid'].mean():.1f}")

print("\n--- GROUNDING VERIFICATION ---")
print(f"  Mean grounding score (template reports): {eval_df['grounding_score'].mean():.3f}")
print(f"  Mean hallucination count per report    : {eval_df['hallucination_count'].mean():.2f}")
print(f"  Note: Template reports are grounded by construction.")
print(f"  The verifier is most informative on LLM-generated free-text reports.")

print("\n--- LIMITATIONS ---")
print("  1. Parser does not handle OR conditions (nationality, position).")
print("  2. FM24 ratings are synthetic -- results are not externally validated.")
print("  3. Tactical phrases outside the attribute vocabulary rely entirely on")
print("     embedding quality, which may not capture football-specific jargon.")
print("  4. ~4.5% of players have no position data.")

print("\n--- FUTURE WORK ---")
print("  1. Fine-tune the embedding model on football-domain text.")
print("  2. Add a cross-encoder reranker for better semantic precision.")
print("  3. Extend the parser to handle OR constraints and comparative queries.")
print("  4. Integrate real-world data (e.g., StatsBomb open data).")
print("  5. Build a Streamlit demo with interactive filter widgets.")
print("=" * 70)